# Experiment Notebook · Fact-Layer Photo QC with Vision-Language Models

Experiment harness for automated quality control of field-installation photos. A vision-language model (VLM) reports **facts** about each photo, attribute by attribute; **plain code** applies the numeric thresholds. There is no pass/fail verdict from the model.

| | |
|---|---|
| **Scope** | one department (`DEPARTMENT = "dept_b"`, section 1); narrow further with `MAIN_CATEGORY`, `SUB_CATEGORY`, `PHASES` |
| **Models** | Kimi K2.5 and Nova Pro on AWS Bedrock; Qwen3-VL 32B (self-hosted vLLM) is wired in but commented out (section 6) |
| **Data** | catalog + photos in a local folder or an object-storage bucket (not included in this repo) |

Catalog `notebook/catalog_fact_phase1.json`: department → main category → sub-category (= one requirement, `FR-…`) → rule → attribute. This public copy ships a small synthetic catalog; the original catalog is proprietary.

For each attribute the model returns `evidence`, `is_true` and, where needed, `value`. The code then checks thresholds (e.g. received signal level ≥ −24 dBm). Because there is no verdict layer yet, an experiment is judged on its facts against the photo folders (`positive` / `negative`).

Sections 1–7: pipeline. 8: change points. 9: experiments, start there.

| Point | What changes | Affects |
|---|---|---|
| 8.1 | Data: photos in the folder / bucket | new baseline |
| 8.2 | Rules of a sub-category: add, remove, edit (`rules=`) | those sub-categories |
| 8.3 | Attributes inside a rule: add, remove, description, type, unit, fields (`rules=`) | those sub-categories |
| 8.4 | Prompt: full template text (`prompt=`, `render=`) | every sub-category |
| 8.5 | Output schema (`schema=`) | every sub-category |
| 8.6 | Threshold of an attribute, checked by code (`rules=`) | those sub-categories |
| 8.7 | Reference photos (`references=`, `max_refs=`, `exclude_self=`, `exact_references=`) | every / one sub-category |
| 8.8 | Model (`models=`) | every sub-category |
| 8.9 | Watermark extraction (`extract=`, `extract_prompt=`, `extract_schema=`) | every photo; the facts are not affected |
| 8.10 | Layer 2: a second look at chosen attributes (`run_layer2(rows, layer2=...)`), off until used | the sub-categories in `layer2` |


In [ ]:
# Environment: Python 3.12+ and `pip install -r requirements.txt` (repo root). Extra libraries mid-session:
# %pip install -q -r ../requirements.txt
#
# Environment variables (copy .env.example to .env in the repo root):
#   AWS_BEARER_TOKEN_BEDROCK, AWS_REGION                           Bedrock access
#   COS_SECRET_ID, COS_SECRET_KEY, COS_REGION, COS_MEDIA_BUCKET    only when STORAGE = "cos"


## 1 · Setup

In [ ]:
import asyncio, base64, copy, io, json, os, pprint, re, time
from collections import defaultdict
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from typing import Annotated, Any, Literal

import pandas as pd
from dotenv import load_dotenv
from PIL import Image, ImageOps
from pydantic import (BaseModel, ConfigDict, Field, StrictFloat, TypeAdapter, ValidationError, create_model,
                      field_validator)

REPO = Path.cwd().parent   # repo root; Jupyter runs a notebook from its own folder, here <repo>/notebook/
load_dotenv(REPO / ".env", override=True)   # .env at the repo root; override=True so keys in .env win over placeholder env vars
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", None)   # show full cell text (requirement, evidence, JSON attribute tables)

# Object-storage bucket and catalog
COS_REGION   = os.environ.get("COS_REGION", "ap-jakarta")
MEDIA_BUCKET = os.environ.get("COS_MEDIA_BUCKET", "<bucket-media>")
# Where the photos live; read when the MANIFEST cell in section 2 runs.
# "cos"   = the object-storage bucket (COS_MEDIA_BUCKET)
# "local" = a folder on disk with EXACTLY the same layout as the bucket (LOCAL_BUCKET: a plain folder or a mounted bucket)
STORAGE      = "local"
LOCAL_BUCKET = REPO / "notebook"                                # STORAGE = "local": photos in notebook/images/ (git-ignored)
CATALOG_PATH = REPO / "notebook" / "catalog_fact_phase1.json"   # sub-categories, rules, attributes (versioned)
assert CATALOG_PATH.is_file(), f"{CATALOG_PATH} not found: run the notebook from <repo>/notebook/"

MODELS = ["kimi-k2-5-bedrock", "nova-pro-bedrock"]                         # Kimi + Nova in one run
# MODELS = ["kimi-k2-5-bedrock", "nova-pro-bedrock", "qwen3-vl-vastai"]    # + Qwen, when the vLLM server is up (section 6)

# Test scope. None = everything. Ids are listed in the sub-category table in section 7.
DEPARTMENT    = "dept_b"   # "dept_a" / "dept_b" / "dept_c" in the original project; the sample catalog only has "dept_b"
MAIN_CATEGORY = None       # main-category id, e.g. "closure-position"
SUB_CATEGORY  = None       # sub-category id (= requirement), e.g. "fr-b-01"
PHASES        = {1}        # sub-category phases to include (1 / 2); None = all, including those with no phase yet

# Model temperature. 0 = deterministic (same photo -> same answer). Output length is not capped.
VERDICT_TEMPERATURE = float(os.environ.get("VERDICT_TEMPERATURE", "0"))


## 2 · Data (object storage bucket or a local folder)

`STORAGE` (section 1): the COS bucket, or a local folder with the same layout (`LOCAL_BUCKET`, by default `notebook/`, so photos live in `notebook/images/`, ignored by git).

    images/<department>/<main_category_id>/<sub_category_id>/positive/<file>    meets the requirement
    images/<department>/<main_category_id>/<sub_category_id>/negative/<file>    has a finding

The catalog file and the photo list are read once, in the `MANIFEST` cell below: after new photos or catalog rules, re-run from the top, or `reload_photos()` mid-experiment.

In [ ]:
class CosStorage:
    """Satu bucket object storage: baca, tulis, dan daftar objek (SDK cos-python-sdk-v5)."""

    def __init__(self, bucket: str, region: str):
        from qcloud_cos import CosConfig, CosS3Client
        self.bucket = bucket
        self.client = CosS3Client(CosConfig(Region=region, SecretId=os.environ["COS_SECRET_ID"],
                                            SecretKey=os.environ["COS_SECRET_KEY"],
                                            Token=os.environ.get("COS_SESSION_TOKEN")))

    def read_bytes(self, key: str) -> bytes:
        return self.client.get_object(Bucket=self.bucket, Key=key)["Body"].get_raw_stream().read()

    def write_bytes(self, key: str, data: bytes) -> None:
        self.client.put_object(Bucket=self.bucket, Key=key, Body=data)

    def list_keys(self, prefix: str) -> list[str]:
        keys, marker = [], ""
        while True:
            resp = self.client.list_objects(Bucket=self.bucket, Prefix=prefix, Marker=marker)
            contents = resp.get("Contents") or []
            keys += [o["Key"] for o in (contents if isinstance(contents, list) else [contents])]
            if str(resp.get("IsTruncated")).lower() != "true":
                return keys
            marker = resp["NextMarker"]


class LocalStorage:
    """Folder biasa dengan susunan yang sama: bucket yang di-mount ke pod (COS CSI), atau folder uji."""

    def __init__(self, root: str):
        self.root = root

    def read_bytes(self, key: str) -> bytes:
        with open(os.path.join(self.root, key), "rb") as f:
            return f.read()

    def write_bytes(self, key: str, data: bytes) -> None:
        path = os.path.join(self.root, key)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "wb") as f:
            f.write(data)

    def list_keys(self, prefix: str) -> list[str]:
        base = os.path.join(self.root, prefix)
        if os.path.isfile(base):
            return [prefix]
        return sorted(os.path.relpath(os.path.join(d, n), self.root).replace(os.sep, "/")
                      for d, _, files in os.walk(base) for n in files)


MEDIA = CosStorage(MEDIA_BUCKET, COS_REGION) if STORAGE == "cos" else LocalStorage(str(LOCAL_BUCKET))

Change point 8.1 tools: `upload_photos` to a sub-category folder, `photo_manifest` (catalog + photos now).

In [ ]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".gif", ".bmp"}


def photo_prefix(department: str, main_category: str, sub_category: str, role: str) -> str:
    """Folder foto satu sub-category di bucket. role = "positive" (lolos) | "negative" (temuan)."""
    assert role in ("positive", "negative"), role
    return f"images/{department}/{main_category}/{sub_category}/{role}/"


def _catalog() -> dict:
    return json.load(open(CATALOG_PATH, encoding="utf-8"))


def _main_of(department: str, sub_category: str, catalog: dict | None = None) -> str:
    """Id main category sebuah sub-category, dicari di katalog."""
    for d in (catalog or _catalog())["departments"]:
        for mc in d["mainCategories"] if d["id"] == department else []:
            if any(c["id"] == sub_category for c in mc["subCategories"]):
                return mc["id"]
    raise ValueError(f"sub-category {department}/{sub_category} tidak ada di catalog")


def upload_photos(files, department: str, sub_category: str, role: str) -> list[str]:
    """Upload foto ke folder sub-category di bucket (main category dicari dari katalog). Nama yang sudah ada tidak ditimpa."""
    prefix = photo_prefix(department, _main_of(department, sub_category), sub_category, role)
    existing = set(MEDIA.list_keys(prefix))
    uploaded = []
    for f in files:
        key = prefix + os.path.basename(str(f))
        if key in existing:
            print("skip, sudah ada:", key)
            continue
        with open(f, "rb") as fh:
            MEDIA.write_bytes(key, fh.read())
        uploaded.append(key)
    print(f"{len(uploaded)} foto di-upload ke {prefix}")
    return uploaded


def photo_manifest() -> dict:
    """Katalog (git) + daftar foto di bucket saat ini: tiap sub-category mendapat `positive`, `negative`, `counts`."""
    manifest = _catalog()
    incomplete = []
    for d in manifest["departments"]:
        for mc in d["mainCategories"]:
            for c in mc["subCategories"]:
                for role in ("positive", "negative"):
                    keys = [k for k in MEDIA.list_keys(photo_prefix(d["id"], mc["id"], c["id"], role))
                            if os.path.splitext(k)[1].lower() in IMAGE_EXTS
                            and not any(part.startswith(".") for part in k.split("/"))]   # .ipynb_checkpoints dll.
                    c[role] = sorted(keys, key=lambda k: os.path.basename(k).lower())
                c["counts"] = {role: len(c[role]) for role in ("positive", "negative")}
                if c.get("rules") and (not c["positive"] or not c["negative"]):
                    incomplete.append(f"{d['id']}/{c['id']} {c['counts']}")
    listed = {k for d in manifest["departments"] for mc in d["mainCategories"] for c in mc["subCategories"]
              for role in ("positive", "negative") for k in c[role]}
    missing = [k for d in manifest["departments"] for mc in d["mainCategories"] for c in mc["subCategories"]
               for role in ("positive", "negative") for k in (c.get("references") or {}).get(role) or []
               if k not in listed and k not in MEDIA.list_keys(k)]
    if missing:
        print("Reference di catalog yang tidak ada di bucket:", *missing, sep="\n  ")
    if incomplete:
        print("Sub-category yang punya rule tapi belum ada foto positive / negative:", *incomplete, sep="\n  ")
    return manifest

The data in use (`MANIFEST`): the catalog + photos as they are when this cell runs; `_load` and `find_category` look things up in memory. Photos added mid-experiment: `reload_photos()`, which keeps `hasil` and prints the new photo keys.

In [ ]:
# Katalog + daftar foto, dibaca sekali. Foto baru / katalog berubah: jalankan ulang dari atas, atau reload_photos().
MANIFEST = photo_manifest()


def _photo_keys() -> set[str]:
    """Semua key foto di MANIFEST."""
    return {k for d in MANIFEST["departments"] for mc in d["mainCategories"] for c in mc["subCategories"]
            for role in ("positive", "negative") for k in c.get(role) or []}


def reload_photos() -> None:
    """Foto baru di tengah eksperimen: baca ulang folder foto + katalog tanpa jalankan ulang dari atas, jadi `hasil` tetap.
    Mencetak key foto yang baru masuk (untuk photos=[...]); photos_of() dan run_exp() tanpa photos= langsung memakainya."""
    global MANIFEST
    before = _photo_keys()
    MANIFEST = photo_manifest()
    now = _photo_keys()
    new, gone = sorted(now - before), sorted(before - now)
    print(f"{len(new)} foto baru", *new, sep="\n  ")
    if gone:
        print(f"{len(gone)} foto sudah tidak ada di folder", *gone, sep="\n  ")


def _load() -> dict:
    """Data yang sedang dipakai (lihat MANIFEST)."""
    return MANIFEST


def find_category(department_id: str, sub_category_id: str) -> tuple[dict | None, dict | None]:
    """Return (main_category, category) untuk (department_id, sub_category_id), atau (None, None)."""
    for dept in _load().get("departments", []):
        if dept["id"] != department_id:
            continue
        for mc in dept.get("mainCategories", []):
            for c in mc["subCategories"]:
                if c["id"] == sub_category_id:
                    return mc, c
    return None, None

In [ ]:
UPLOAD_MAX_PX = 1280            # foto yg DINILAI - = frontend RESIZE_MAX_PX (frontend/src/lib/api/image.ts)
REF_MAX_PX = 768               # foto referensi
JPEG_QUALITY = 85


def _encode_pil(img: Image.Image, max_px: int, quality: int = JPEG_QUALITY) -> str:
    """PIL image → base64 JPEG; diputar sesuai EXIF Orientation, lalu resize proporsional ke max_px
    (fit inside, NEVER enlarge)."""
    img = ImageOps.exif_transpose(img).convert("RGB")   # foto HP: Orientation 6/8 -> tegak
    img.thumbnail((max_px, max_px))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=quality)
    return base64.b64encode(buf.getvalue()).decode("ascii")


def encode_photo(key: str, max_px: int = UPLOAD_MAX_PX) -> tuple[str, str]:
    """Foto yang dinilai, dibaca dari bucket -> (b64, 'image/jpeg'). Resize sama dengan encode_file di backend."""
    return _encode_pil(Image.open(io.BytesIO(MEDIA.read_bytes(key))), max_px), "image/jpeg"

In [ ]:
@dataclass
class RefImage:
    base64: str
    mime_type: str
    role: str                        # "positive" | "negative"
    label: str = ""
    url: str = ""                    # /refimg URL (utk ditampilkan di hasil; provider abaikan, pakai base64)


def _encode(rel_path: str) -> str | None:
    try:
        return _encode_pil(Image.open(io.BytesIO(MEDIA.read_bytes(rel_path))), REF_MAX_PX)
    except Exception as e:
        print(f"[references] skip {rel_path}: {e}")
        return None

## 3 · Rules, attributes and the prompt

A sub-category holds rules (`title`, `description`, `attributes`); an attribute (`name`, `type`, `description`, `unit`, `fields`, `threshold`) is what the AI is asked. Its answer per attribute:

| `type` | Answer | Example |
|---|---|---|
| `bool` | `evidence`, `is_true` | `retak` → `is_true: false` |
| `text` | + `value`: the text read | `label_partner_terlihat` → "PARTNER-X" |
| `number` | + `value`: the number, in `unit` | `kecepatan_download` → 102.7 |
| `json` | + `value`: one item per object seen, each with the `fields` | device labels → `[{"kode": "…", "format_sesuai": true}, …]` |

- `name` is the field name in the answer (`snake_case`); the same name in two rules is asked once.
- `is_true` = the attribute `description` holds for the photo (`screenshot_ookla` true = from Ookla, `retak` true = cracked); false when it does not, cannot be seen, or is unsure, and `value` is then null. True is not "good": write a defect as seen ("retak", not "tidak retak"), and give "the object is visible" its own attribute.
- `evidence` comes first, so the model looks before it answers; the code does not use it.
- No verdict, reason or recommendation for now; a `threshold` is not in the prompt, the code checks it (section 5).
- Rules come from the catalog or `rules=` (8.2, 8.3); `render_rules` numbers them in the prompt.

In [ ]:
def active_rules(department: str, sub_category: str, rules: dict | None = None) -> list:
    """Rule yang berlaku: `rules` (setelan Exp) bila ada untuk sub-category ini, kalau tidak salinan rule di MANIFEST
    (katalog; mengubah salinannya tidak mengubah MANIFEST)."""
    key = f"{department}/{sub_category}"
    if rules and key in rules:
        return rules[key]
    _, c = find_category(department, sub_category)
    return copy.deepcopy((c or {}).get("rules") or [])


def attributes_of(rules: list) -> dict:
    """Semua atribut dari rule, urut kemunculan; atribut yang dipakai beberapa rule cukup ditanya sekali."""
    out = {}
    for r in rules:
        for a in r.get("attributes") or []:
            out.setdefault(a["name"], a)
    return out


def rule_of(rules: list, title: str) -> dict:
    """Rule dengan title itu, untuk diedit di sel Exp: rule_of(R, "Bentuk coran")["attributes"].append({...})."""
    for r in rules:
        if r.get("title") == title:
            return r
    raise KeyError(f"rule {title!r} tidak ditemukan; yang ada: {[r.get('title') for r in rules]}")


def _attribute_line(a: dict) -> str:
    extra = {"text": " (tulis teks yang terbaca di value)",
             "number": f" (tulis angkanya di value, satuan {a.get('unit', '')})",
             "json": " (value: daftar, satu item per objek yang terlihat; tiap item: "
                     + "; ".join(f"{f['name']} = {f['description']}" for f in a.get("fields") or []) + ")"}
    return f"  - {a['name']}: {a['description']}{extra.get(a['type'], '')}"


def render_rules(rules: list) -> str:
    """Rule -> teks {rules} di prompt: nomor urut + title, description, lalu atributnya. threshold tidak ditulis:
    AI hanya melaporkan fakta, threshold dicek kode (bagian 5)."""
    lines, asked = [], set()
    for i, r in enumerate(rules, 1):
        lines.append(f"Rule {i} · {r['title']}")
        if r.get("description"):
            lines.append(f"  {r['description']}")
        for a in r.get("attributes") or []:
            lines.append(f"  - {a['name']}: (sama dengan di atas)" if a["name"] in asked else _attribute_line(a))
            asked.add(a["name"])
    return "\n".join(lines)


def render_prompt(template: str, sub: dict, rules: list, render=None) -> str:
    """Isi template untuk satu sub-category: {sub_category_id} = id (FR-A-03), {requirement} = kalimat requirement
    (= sub-category itu), {rules} = rule + atribut (render_rules, atau fungsi `render` dari setelan Exp)."""
    return (template.replace("{sub_category_id}", sub["id"].upper())
            .replace("{requirement}", sub.get("requirement") or "").replace("{rules}", (render or render_rules)(rules)))

In [ ]:
# Example: the rules and attributes of FR-B-01 as they appear in the prompt
print(render_rules(active_rules("dept_b", "fr-b-01")))


In [ ]:
PROMPTS = {}
PROMPTS["facts"] = """Anda auditor QC instalasi jaringan telekomunikasi. Sub-category yang dinilai: {sub_category_id}.
Requirement: {requirement}

Rule untuk sub-category ini, masing-masing dengan atribut yang harus diamati:
{rules}

Gambar referensi (bila ada) adalah contoh foto sub-category ini: "positive" = sesuai requirement, "negative" = ada temuan. Pakai sebagai pembanding untuk memahami objek dan kondisi yang dimaksud. Yang dinilai tetap FOTO YANG DINILAI.

Tugas: amati FOTO YANG DINILAI dan laporkan SETIAP atribut di atas, tanpa menilai lulus / gagal:
- evidence: tulis DULU apa yang terlihat di foto untuk atribut itu, singkat dan spesifik. Boleh menyebut referensi yang mirip (mis. "mirip Ref 2 (negative)").
- is_true: true bila pernyataan di deskripsi atribut itu BENAR untuk foto ini; false bila tidak benar, tidak terlihat, atau tidak bisa dipastikan. true tidak berarti baik: untuk atribut yang menyebut cacat atau kondisi buruk (mis. "retak"), is_true = true berarti cacat itu TERLIHAT.
- value: untuk atribut yang meminta nilai: teks atau angka yang TERBACA persis di foto (mis. nomor label, hasil ukur); untuk atribut berupa daftar (mis. beberapa label perangkat): satu item per objek yang terlihat, dengan field yang diminta. null bila is_true false atau tidak terbaca (daftar: [] bila tidak ada). Jangan menebak.

Kalibrasi:
- Jangan mengarang temuan saat ragu; tandai menyimpang hanya bila JELAS terlihat.

Gambar referensi mungkin diberi markup auditor (panah, lingkaran, teks); fokus pada markup itu saat membandingkan. Apabila terdapat 2 gambar before dan after bersebelahan, evaluasi gambar after"""

# Layer 2 (point 8.10, mati sampai dipakai): template cek ulang = PROMPTS["facts"] + satu baris kalibrasi. {rules} di sini
# hanya berisi atribut yang dicek ulang. Template sendiri per sub-category: "prompt" di setelan layer2.
PROMPTS["layer2"] = PROMPTS["facts"].replace(
    "Kalibrasi:\n",
    "Kalibrasi:\n- CEK ULANG: atribut di atas sering salah baca. Amati FOTO YANG DINILAI sekali lagi dengan cermat, "
    "bandingkan dengan referensi positive DAN negative, dan jawab hanya dari yang benar-benar terlihat.\n")
assert PROMPTS["layer2"] != PROMPTS["facts"], "baris Kalibrasi tidak ditemukan di PROMPTS['facts']"

In [ ]:
# Ekstraksi watermark (point 8.9): call kedua, foto saja tanpa referensi. Nova menambahkan
# "\n\nPanggil tool emit_extraction dengan hasilnya." di belakangnya (bagian 6).
PROMPTS["extract"] = """Baca TEKS WATERMARK / overlay GPS-camera yang TERCETAK di foto ini. Ekstrak 4 field:
- captured_at: tanggal+waktu watermark → ISO 8601 "YYYY-MM-DDTHH:MM:SS". Tangani variasi: "14 Jan 2025 13.14.42" → "2025-01-14T13:14:42"; tanggal ANGKA = Indonesia HARI-dulu DD/MM/YYYY ("20/04/2026" → "2026-04-20T00:00:00", JANGAN gaya US bulan-dulu); nama bulan Indonesia ("24 Maret 2026" → "2026-03-24T00:00:00", Mei=05/Agustus=08/Okt=10); ada tanggal TANPA jam → jam "00:00:00". HANYA jam tanpa tanggal / tak terbaca → null.
- latitude_text / longitude_text: teks LINTANG dan BUJUR masing-masing PERSIS seperti tercetak di watermark, lengkap dengan tanda minus, simbol, dan huruf arahnya, format apa pun: DMS (mis. 6°14'18,33"S dan 106°46'38,364"E) atau desimal (mis. -6.123456° dan 106.123456°). JANGAN dikonversi, dihitung, atau dibulatkan. DILARANG meng-geocode / menebak dari nama tempat/alamat. Bila watermark cuma berisi ALAMAT tanpa angka koordinat → null.
- address: teks alamat watermark apa adanya (jalan/kecamatan/kota/provinsi). null bila tak ada.

KERAS: keempat field HANYA dari teks/angka yang BENAR-BENAR TERCETAK & TERBACA di watermark. Bila foto TIDAK punya watermark sama sekali → KEEMPATNYA null. JANGAN menebak/mengarang/geocode. Lebih baik null daripada salah."""

## 4 · Output schema

One required field per attribute, shaped by its `type`, and nothing else. `bool` / `text` / `number` each have one fixed shape; a `json` attribute's items are built from its own `fields` (e.g. `kode` + `format_sesuai`), so it must list them (worked example: FR-B-03 in the sample catalog). To change it, pass a renamed copy of `build_output_model` as `schema=` (8.5).

In [ ]:
# evidence di depan: model menulis apa yang terlihat dulu, baru menjawab (structured output diisi berurutan)
IS_TRUE = "true bila pernyataan di deskripsi atribut ini BENAR untuk foto; false bila tidak benar atau tidak bisa dipastikan"


class FactBool(BaseModel):
    evidence: str = Field(..., description="apa yang terlihat sebagai dasar jawaban")
    is_true: bool = Field(..., description=IS_TRUE)


class FactText(BaseModel):
    evidence: str = Field(..., description="apa yang terlihat sebagai dasar jawaban")
    is_true: bool = Field(..., description=IS_TRUE)
    value: str | None = Field(..., description="teks yang terbaca persis; null bila is_true false atau tidak terbaca")


class FactNumber(BaseModel):
    evidence: str = Field(..., description="apa yang terlihat sebagai dasar jawaban")
    is_true: bool = Field(..., description=IS_TRUE)
    value: float | None = Field(..., description="angka yang terbaca; null bila is_true false atau tidak terbaca")


_MODEL_CACHE: dict = {}
_ITEM_KIND = {"bool": bool | None, "text": str | None, "number": float | None}


def _json_model(a: dict) -> type[BaseModel]:
    """Atribut json: value = daftar item, satu per objek yang terlihat; tiap item punya `fields` atribut itu.
    Contoh: label_perangkat, fields kode (text) + format_sesuai (bool) -> value = [{"kode": "ABC-001", "format_sesuai": true}, …]."""
    item = create_model(f"Item_{a['name']}", **{
        f["name"]: (_ITEM_KIND[f["type"]],
                    Field(..., description=f["description"] + (f" (satuan {f['unit']})" if f.get("unit") else "")
                          + "; null bila tidak terbaca"))
        for f in a["fields"]})
    return create_model(f"Fact_{a['name']}", evidence=(str, Field(..., description="apa yang terlihat sebagai dasar jawaban")),
                        is_true=(bool, Field(..., description=IS_TRUE)),
                        value=(list[item], Field(..., description="satu item per objek yang terlihat; [] bila tidak ada")))


def build_output_model(attributes: dict) -> type[BaseModel]:
    """Schema output untuk satu sub-category: satu field per atribut (bool / text / number / json), tanpa verdict."""
    key = json.dumps(attributes, sort_keys=True, ensure_ascii=False)
    if key in _MODEL_CACHE:
        return _MODEL_CACHE[key]
    fixed = {"bool": FactBool, "text": FactText, "number": FactNumber}   # bentuk jawabannya sama untuk atribut mana pun
    fields = {}
    for k, a in attributes.items():
        desc = a["description"] + (f" (satuan {a['unit']})" if a.get("unit") else "")
        if a["type"] == "json":
            # isi tiap item beda per atribut (mis. label_perangkat: kode + format_sesuai), jadi model-nya dibangun dari
            # `fields` yang WAJIB ditulis di atribut itu. Contoh lengkap: FR-B-03 di catalog contoh.
            model = _json_model(a)
        else:
            model = fixed[a["type"]]
        fields[k] = (model, Field(..., description=desc))
    _MODEL_CACHE[key] = create_model("OutputModel", **fields)
    return _MODEL_CACHE[key]

The schema as JSON Schema (what Bedrock receives), and the answer read back.

In [ ]:
@dataclass
class Facts:
    facts: dict | None               # jawaban model per atribut; None bila output tidak valid
    raw_text: str = ""
    token_usage: dict | None = None


# ── JSON Schema derivation (flat, no $ref) ─────────────────────────────────
def _inline(node: Any, defs: dict) -> Any:
    if isinstance(node, dict):
        if "$ref" in node:
            merged = _inline({k: v for k, v in defs.get(node["$ref"].split("/")[-1], {}).items()}, defs)
            for k, v in node.items():
                if k != "$ref":
                    merged[k] = _inline(v, defs)
            return merged
        if isinstance(node.get("allOf"), list) and len(node["allOf"]) == 1 and "$ref" in node["allOf"][0]:
            base = _inline(node["allOf"][0], defs)
            for k, v in node.items():
                if k != "allOf":
                    base[k] = _inline(v, defs)
            return base
        return {k: _inline(v, defs) for k, v in node.items()}
    if isinstance(node, list):
        return [_inline(v, defs) for v in node]
    return node


def _strip(node: Any) -> None:
    if isinstance(node, dict):
        node.pop("title", None)
        node.pop("default", None)
        for v in node.values():
            _strip(v)
    elif isinstance(node, list):
        for v in node:
            _strip(v)


def flat_schema(model: type[BaseModel]) -> dict:
    s = model.model_json_schema()
    defs = s.pop("$defs", {})
    s = _inline(s, defs)
    _strip(s)
    return s


def _set_additional_false(node: Any) -> None:
    if isinstance(node, dict):
        if node.get("type") == "object":
            node["additionalProperties"] = False
        for v in node.values():
            _set_additional_false(v)
    elif isinstance(node, list):
        for v in node:
            _set_additional_false(v)


def bedrock_schema(model: type[BaseModel]) -> dict:
    s = flat_schema(model)
    _set_additional_false(s)
    return s


# ── Validasi balik ─────────────────────────────────────────────────────────
def parse_json(model: type[BaseModel], text: str) -> BaseModel | None:
    if not text:
        return None
    try:
        return model.model_validate_json(text)
    except (ValidationError, ValueError):
        start, end = text.find("{"), text.rfind("}")
        if start != -1 and end > start:
            try:
                return model.model_validate_json(text[start:end + 1])
            except (ValidationError, ValueError):
                return None
    return None


def parse_dict(model: type[BaseModel], d: dict) -> BaseModel | None:
    try:
        return model.model_validate(d or {})
    except ValidationError:
        return None

Watermark extraction schema (8.9): four fields (`captured_at`, `latitude_text`, `longitude_text`, `address`), each `null` when not printed on the photo. The model copies latitude and longitude as printed; `to_decimal` turns each into a signed decimal `latitude` / `longitude` (decimal kept as is, DMS = degrees + minutes/60 + seconds/3600, S / W negative).

In [ ]:
@dataclass
class Extraction:
    """Hasil ekstraksi watermark dari Provider.extract() (call sendiri, bukan menempel di call fakta). Menempel
    di call fakta dulu bikin Qwen ngarang koordinat + sedikit menyenggol fakta; call terpisah -> literal + fakta bersih.
    """
    captured_at: str | None = None         # string ISO 8601 dari watermark (parse ke datetime di hilir)
    latitude_text: str | None = None       # teks lintang watermark apa adanya (DMS atau desimal)
    longitude_text: str | None = None      # teks bujur watermark apa adanya
    latitude: float | None = None          # derajat desimal bertanda (S negatif), dari to_decimal
    longitude: float | None = None         # derajat desimal bertanda (W negatif), dari to_decimal
    address: str | None = None             # teks alamat watermark apa adanya
    token_usage: dict | None = None        # token call ekstraksi (terpisah dari call fakta)


# ── Ekstraksi watermark (schema + parser TERPISAH, untuk mode SEPARATE) ─────
_EXTRACTION_MODEL: type[BaseModel] | None = None


def watermark_model() -> type[BaseModel]:
    """Model 4-field ekstraksi watermark untuk call TERPISAH (mode separate). Field(...) (required-nullable)
    krn OpenAI strict wajib semua key di 'required'; nullability lewat tipe opsional."""
    global _EXTRACTION_MODEL
    if _EXTRACTION_MODEL is None:
        _EXTRACTION_MODEL = create_model(
            "ExtractionModel",
            captured_at=(str | None, Field(..., description="Tanggal+waktu watermark, ISO 8601 'YYYY-MM-DDTHH:MM:SS'; jam 00:00:00 bila ada tanggal tanpa jam. null bila tak ada/tak terbaca.")),
            latitude_text=(str | None, Field(..., description="Teks LINTANG watermark PERSIS seperti tercetak (DMS atau desimal, dengan tanda / huruf arahnya), tanpa dikonversi. null bila tak ada angka koordinat.")),
            longitude_text=(str | None, Field(..., description="Teks BUJUR watermark PERSIS seperti tercetak (DMS atau desimal, dengan tanda / huruf arahnya), tanpa dikonversi. null bila tak ada angka koordinat.")),
            address=(str | None, Field(..., description="Teks alamat watermark apa adanya. null bila tak ada.")),
        )
    return _EXTRACTION_MODEL


# Arah negatif: tanda minus; huruf besar S / W / LS / BB yang tidak menempel ke huruf lain (boleh menempel ke angka:
# 18.33S, S6°); huruf kecil s / w tepat setelah simbol (18.33"s); kata Selatan / Barat / South / West. Huruf kecil
# setelah angka (6d14m18.33s) = satuan derajat / menit / detik, bukan arah.
_NEGATIVE = re.compile(r"[-−]|(?<![A-Za-z])(?:S|W|LS|BB)(?![A-Za-z])|(?<=[\"″”'′’°º])[sw](?![A-Za-z])"
                       r"|(?i:\b(?:selatan|barat|south|west)\b)")


def to_decimal(text: str | None, limit: int) -> float | None:
    """Teks satu komponen koordinat watermark -> derajat desimal bertanda (limit: 90 lintang, 180 bujur).
    Format dikenali dari JUMLAH ANGKA, bukan simbolnya: 1 = desimal (-6.123456°), 2 = derajat + menit (6°14,3055'),
    3 = DMS (6°14'18,33"S); nilai = derajat + menit/60 + detik/3600. Hanya angka terakhir boleh berdesimal (titik atau
    koma) dan menit / detik < 60: teks rancu (mis. 6,14,18) -> None, tidak ditebak. Negatif bila ada tanda minus atau
    arah Selatan / Barat (_NEGATIVE). Di luar rentang -> None. Dibulatkan 8 angka di belakang koma (~1 mm), lebih teliti
    dari watermark mana pun (0,001" ≈ 3 cm; 6 angka desimal ≈ 11 cm), jadi presisi aslinya tidak hilang."""
    if not text:
        return None
    numbers = re.findall(r"\d+(?:[.,]\d+)?", text)
    if not 1 <= len(numbers) <= 3 or any(re.search(r"[.,]", n) for n in numbers[:-1]):
        return None
    parts = [float(n.replace(",", ".")) for n in numbers]
    if any(p >= 60 for p in parts[1:]):
        return None
    value = sum(p / 60 ** i for i, p in enumerate(parts))
    if _NEGATIVE.search(text):
        value = -value
    return round(value, 8) if abs(value) <= limit else None


def to_extraction(vm: BaseModel | None, token_usage: dict | None = None) -> Extraction:
    """Normalisasi instance ExtractionModel (dinamis) → Extraction carrier; latitude / longitude dihitung kode dari
    teks yang disalin model (to_decimal), bukan oleh model."""
    if vm is None:
        return Extraction(token_usage=token_usage)
    latitude_text, longitude_text = getattr(vm, "latitude_text", None), getattr(vm, "longitude_text", None)
    return Extraction(
        captured_at=getattr(vm, "captured_at", None),
        latitude_text=latitude_text,
        longitude_text=longitude_text,
        latitude=to_decimal(latitude_text, 90),
        longitude=to_decimal(longitude_text, 180),
        address=getattr(vm, "address", None),
        token_usage=token_usage,
    )

## 5 · Threshold check

For every attribute with a `threshold`, the code checks: `is_true`, the value read, and every condition met (`>`, `>=`, `<`, `<=`, `=`; only a `number` attribute has one, a count is its own `number` attribute). The AI never sees it. Results, kept for the future verdict: `thresholdChecks` (value, condition, met or not) and `thresholdFailed` (attribute names not met). A threshold change is free with `reevaluate` (8.6). `check_rules` validates the rule format before any model call, so a typo costs nothing.

In [ ]:
LIMITS = {">": lambda v, x: v > x, ">=": lambda v, x: v >= x, "<": lambda v, x: v < x, "<=": lambda v, x: v <= x,
          "=": lambda v, x: v == x}


def threshold_ok(a: dict, fact: dict | None) -> bool:
    """Atribut ber-threshold (hanya number) memenuhi syaratnya: is_true, angkanya terbaca, dan semua kondisi terpenuhi."""
    if not fact or not fact.get("is_true") or fact.get("value") is None:
        return False
    return all(LIMITS[op](fact["value"], x) for op, x in a["threshold"].items())


def check_thresholds(rules: list, facts: dict | None) -> list:
    """Cek threshold tiap atribut yang punya threshold (dicek kode, bukan AI). Belum ada verdict: hasilnya dicatat saja."""
    if facts is None:
        return []
    return [{"name": k, "value": (facts.get(k) or {}).get("value"), "threshold": a["threshold"],
             "ok": threshold_ok(a, facts.get(k))}
            for k, a in attributes_of(rules).items() if a.get("threshold")]


# ── Format rule: dicek Pydantic sebelum model dipanggil ─────────────────────
# Satu class per type atribut. extra="forbid" menolak key yang tidak dikenal, jadi salah ketik ("treshold"),
# threshold di luar number, dan fields di luar json otomatis ditolak.
Op = Literal[tuple(LIMITS)]   # > >= < <= =


class _Spec(BaseModel):
    model_config = ConfigDict(extra="forbid")


class ItemField(_Spec):
    """Satu field (kolom) di tiap item atribut json, mis. kode / format_sesuai."""
    name: str = Field(min_length=1)
    type: Literal["bool", "text", "number"]
    description: str = Field(min_length=1)
    unit: str | None = None


class _Attribute(_Spec):
    name: str = Field(min_length=1)
    description: str = Field(min_length=1)
    unit: str | None = None   # satuan angka (dBm, Mbps, ...), opsional: hanya ditempel ke description
    expected: bool = True     # jawaban yang lolos: true, atau false untuk cacat (mis. retak); dipakai ground truth, tidak masuk prompt


class BoolAttribute(_Attribute):
    type: Literal["bool"]


class TextAttribute(_Attribute):
    type: Literal["text"]


class NumberAttribute(_Attribute):
    type: Literal["number"]
    threshold: dict[Op, StrictFloat] | None = None   # Strict: -24 / -24.5 diterima, "-24" (teks) dan true ditolak


class JsonAttribute(_Attribute):
    type: Literal["json"]
    fields: list[ItemField] = Field(min_length=1)

    @field_validator("fields")
    @classmethod
    def _unique_names(cls, fields: list[ItemField]) -> list[ItemField]:
        names = [f.name for f in fields]
        if len(names) != len(set(names)):
            raise ValueError(f"name field duplikat: {names}")
        return fields


Attribute = Annotated[BoolAttribute | TextAttribute | NumberAttribute | JsonAttribute, Field(discriminator="type")]


class Rule(_Spec):
    title: str = Field(min_length=1)
    description: str | None = None
    attributes: list[Attribute] = Field(min_length=1)


_RULES = TypeAdapter(list[Rule])   # semua rule satu sub-category; tiap rule dicek dengan class Rule


def _across_rules(rules: list[Rule]) -> list[str]:
    """Aturan lintas rule (class Rule hanya melihat dirinya sendiri): title unik, satu nama atribut = satu type."""
    titles = [r.title for r in rules]
    problems = [f"title rule duplikat: {titles}"] if len(titles) != len(set(titles)) else []
    types = {}
    for a in (a for r in rules for a in r.attributes):
        if types.setdefault(a.name, a.type) != a.type:
            problems.append(f"{a.name}: dipakai dengan type berbeda ({types[a.name]} dan {a.type})")
    return problems


def check_rules(department: str, sub_category: str, rules: list) -> None:
    """Cek format rule sebelum model dipanggil, supaya salah ketik di sel Exp ketahuan sebelum ada biaya.
    Lokasi error: index rule . attributes . index atribut . type . key, mis. 0.attributes.2.number.threshold."""
    try:
        problems = _across_rules(_RULES.validate_python(rules))
    except ValidationError as e:
        problems = [f"{'.'.join(map(str, x['loc'])) or 'rules'}: {x['msg']}" for x in e.errors(include_url=False)]
    if problems:
        raise ValueError(f"rule {department}/{sub_category} tidak valid:\n  " + "\n  ".join(problems))

In [ ]:
# Example without a model call: FR-B-02 (optical power reading).
# wavelength 1490 meets "= 1490", but -25.3 dBm does not meet ">= -24"
contoh_fakta = {"hasil_opm_terlihat": {"is_true": True}, "panjang_gelombang": {"is_true": True, "value": 1490},
                "nilai_rsl": {"is_true": True, "value": -25.3}}
check_thresholds(active_rules("dept_b", "fr-b-02"), contoh_fakta)


## 6 · Models

Message: prompt → references (`Ref k (role) — file:`) → `Foto yang dinilai:` → photo; the answer follows section 4. Watermark extraction (8.9) is a second, parallel call with `PROMPTS["extract"]` and the photo only (`extract=False` turns it off).

In [ ]:
def _json_output(model: type[BaseModel], name: str) -> dict:
    """outputConfig structured output Bedrock (grammar/constrained decoding, setara vLLM json_schema)."""
    return {"textFormat": {"type": "json_schema", "structure": {"jsonSchema": {
        "schema": json.dumps(bedrock_schema(model)), "name": name}}}}


def _text_of(resp: dict) -> str:
    return "".join(b.get("text", "") for b in resp["output"]["message"]["content"] if "text" in b)


def _content(prompt_text: str, references: list, image_b64: str, mime_type: str, last: str) -> list:
    content = [{"text": prompt_text}]
    for i, r in enumerate(references):
        content += [
            {"text": f"Ref {i + 1} ({r.role}) — {r.label}:"},
            {"image": {"format": r.mime_type.split("/")[-1], "source": {"bytes": base64.b64decode(r.base64)}}},
        ]
    return content + [{"text": last},
                      {"image": {"format": mime_type.split("/")[-1], "source": {"bytes": base64.b64decode(image_b64)}}}]


class KimiBedrockProvider:
    def __init__(self, label: str):
        self.label = label
        self.region = os.environ.get("AWS_REGION", "ap-southeast-3")
        self.model = os.environ.get("KIMI_MODEL_ID", "moonshotai.kimi-k2.5")

    def _client(self):
        import boto3
        from botocore.config import Config
        # Structured output: schema baru dikompilasi jadi grammar dulu di Bedrock (bisa beberapa menit di
        # call pertama per schema), read_timeout default 60 dtk tidak cukup.
        return boto3.client("bedrock-runtime", region_name=self.region,
                            config=Config(retries={"max_attempts": 10, "mode": "adaptive"}, read_timeout=300))

    async def analyze(self, image_b64: str, mime_type: str, references: list[RefImage], prompt_text: str,
                      schema_model: type[BaseModel]) -> Facts:
        """Fakta per atribut untuk satu foto (structured output = schema_model)."""
        resp = await asyncio.to_thread(
            self._client().converse, modelId=self.model,
            messages=[{"role": "user", "content": _content(prompt_text, references, image_b64, mime_type, "Foto yang dinilai:")}],
            inferenceConfig={"maxTokens": 4096, "temperature": VERDICT_TEMPERATURE},
            outputConfig=_json_output(schema_model, "facts"))
        u = resp.get("usage", {})
        text = _text_of(resp)
        vm = parse_json(schema_model, text)
        return Facts(vm.model_dump() if vm else None, text,
                     {"promptTokens": u.get("inputTokens", 0), "outputTokens": u.get("outputTokens", 0)})

    async def extract(self, image_b64: str, mime_type: str, prompt_text: str | None = None, schema_model=None) -> Extraction:
        """Ekstraksi watermark - CALL TERPISAH (1 foto, tanpa ref, prompt fokus) via structured output (outputConfig)."""
        import boto3
        from botocore.config import Config

        # Retry bawaan boto3; read_timeout 300 karena schema baru dikompilasi dulu di call pertama (lihat _client).
        client = boto3.client("bedrock-runtime", region_name=self.region, config=Config(read_timeout=300))
        model = schema_model or watermark_model()
        content = [
            {"text": prompt_text or PROMPTS["extract"]},
            {"image": {"format": mime_type.split("/")[-1], "source": {"bytes": base64.b64decode(image_b64)}}},
        ]
        resp = await asyncio.to_thread(
            client.converse,
            modelId=self.model,
            messages=[{"role": "user", "content": content}],
            inferenceConfig={"maxTokens": 1024, "temperature": VERDICT_TEMPERATURE},
            outputConfig=_json_output(model, "extraction"),
        )
        u = resp.get("usage", {})
        tu = {"promptTokens": u.get("inputTokens", 0), "outputTokens": u.get("outputTokens", 0)}
        return to_extraction(parse_json(model, _text_of(resp)), tu)

In [ ]:
class NovaBedrockProvider:
    def __init__(self, label: str):
        self.label = label
        self.region = os.environ.get("AWS_REGION", "ap-southeast-3")
        self.model = os.environ.get("NOVA_MODEL_ID", "amazon.nova-pro-v1:0")
        self._client = None

    def _client_get(self):
        # Satu client dipakai ulang lintas call; mode "adaptive" = retry otomatis saat ThrottlingException
        # (rate limit Nova Pro on-demand Jakarta ketat).
        if self._client is None:
            import boto3
            from botocore.config import Config
            self._client = boto3.client(
                "bedrock-runtime",
                region_name=self.region,
                config=Config(retries={"max_attempts": 10, "mode": "adaptive"}, read_timeout=120),
            )
        return self._client

    async def analyze(self, image_b64: str, mime_type: str, references: list[RefImage], prompt_text: str,
                      schema_model: type[BaseModel]) -> Facts:
        """Fakta per atribut untuk satu foto (tool use, inputSchema = schema_model)."""
        content = _content(prompt_text, references, image_b64, mime_type,
                           "Foto yang dinilai. Panggil tool emit_facts dengan hasilnya.")
        resp = await asyncio.to_thread(
            self._client_get().converse,
            modelId=self.model,
            messages=[{"role": "user", "content": content}],
            toolConfig={
                "tools": [{"toolSpec": {
                    "name": "emit_facts",
                    "description": "Laporkan fakta per atribut untuk foto yang dinilai.",
                    "inputSchema": {"json": bedrock_schema(schema_model)},
                }}],
                "toolChoice": {"tool": {"name": "emit_facts"}},
            },
            inferenceConfig={"maxTokens": 4096, "temperature": VERDICT_TEMPERATURE},
        )
        j = {}
        for block in resp["output"]["message"]["content"]:
            if "toolUse" in block:
                j = block["toolUse"]["input"]
        vm = parse_dict(schema_model, j)
        u = resp.get("usage", {})
        return Facts(vm.model_dump() if vm else None, str(j),
                     {"promptTokens": u.get("inputTokens", 0), "outputTokens": u.get("outputTokens", 0)})

    async def extract(self, image_b64: str, mime_type: str, prompt_text: str | None = None, schema_model=None) -> Extraction:
        """Ekstraksi watermark - CALL TERPISAH (1 foto, tanpa ref, prompt fokus) via tool-use."""
        client = self._client_get()
        model = schema_model or watermark_model()
        content = [
            {"text": (prompt_text or PROMPTS["extract"]) + "\n\nPanggil tool emit_extraction dengan hasilnya."},
            {"image": {"format": mime_type.split("/")[-1], "source": {"bytes": base64.b64decode(image_b64)}}},
        ]
        resp = await asyncio.to_thread(
            client.converse,
            modelId=self.model,
            messages=[{"role": "user", "content": content}],
            toolConfig={
                "tools": [{"toolSpec": {
                    "name": "emit_extraction",
                    "description": "Laporkan field watermark: captured_at, latitude_text, longitude_text, address.",
                    "inputSchema": {"json": bedrock_schema(model)},
                }}],
                "toolChoice": {"tool": {"name": "emit_extraction"}},
            },
            inferenceConfig={"maxTokens": 1024, "temperature": VERDICT_TEMPERATURE},
        )
        j = {}
        for block in resp["output"]["message"]["content"]:
            if "toolUse" in block:
                j = block["toolUse"]["input"]
        u = resp.get("usage", {})
        tu = {"promptTokens": u.get("inputTokens", 0), "outputTokens": u.get("outputTokens", 0)}
        return to_extraction(parse_dict(model, j), tu)

**Qwen3-VL 32B (self-hosted vLLM), commented out.** Needs `VLLM_BASE_URL`, `VLLM_MODEL`, `VLLM_API_KEY` in `.env`. To turn it on, uncomment the cell below, the `qwen3-vl-vastai` line in `MODEL`, and the second `MODELS` line in section 1.

In [ ]:
# class QwenVllmProvider:
#     def __init__(self, label: str, cfg: dict | None = None):
#         cfg = cfg or {}
#         self.label = label
#         self.base_url = os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1")
#         self.served = cfg.get("served") or os.environ.get("VLLM_MODEL", "qwen3-vl-32b")

#     async def analyze(self, image_b64: str, mime_type: str, references: list[RefImage], prompt_text: str,
#                       schema_model: type[BaseModel]) -> Facts:
#         from openai import AsyncOpenAI

#         client = AsyncOpenAI(base_url=self.base_url, api_key=os.environ.get("VLLM_API_KEY", "EMPTY"))
#         content = [{"type": "text", "text": prompt_text}]
#         for i, r in enumerate(references):
#             content += [
#                 {"type": "text", "text": f"Ref {i + 1} ({r.role}) — {r.label}:"},
#                 {"type": "image_url", "image_url": {"url": f"data:{r.mime_type};base64,{r.base64}"}},
#             ]
#         content += [
#             {"type": "text", "text": "Foto yang dinilai:"},
#             {"type": "image_url", "image_url": {"url": f"data:{mime_type};base64,{image_b64}"}},
#         ]
#         resp = await client.chat.completions.create(
#             model=self.served,
#             temperature=VERDICT_TEMPERATURE,
#             messages=[{"role": "user", "content": content}],
#             response_format={"type": "json_schema", "json_schema": {"name": "facts", "schema": flat_schema(schema_model)}},
#         )
#         raw = resp.choices[0].message.content or "{}"
#         vm = parse_json(schema_model, raw)
#         u = resp.usage
#         token_usage = {"promptTokens": u.prompt_tokens, "outputTokens": u.completion_tokens} if u else None
#         return Facts(vm.model_dump() if vm else None, raw, token_usage)

#     async def extract(self, image_b64: str, mime_type: str, prompt_text: str | None = None, schema_model=None) -> Extraction:
#         """Ekstraksi watermark - CALL TERPISAH (1 foto, tanpa ref, prompt fokus) → Qwen literal (gak ngarang)."""
#         from openai import AsyncOpenAI

#         client = AsyncOpenAI(base_url=self.base_url, api_key=os.environ.get("VLLM_API_KEY", "EMPTY"))
#         content = [
#             {"type": "text", "text": prompt_text or PROMPTS["extract"]},
#             {"type": "image_url", "image_url": {"url": f"data:{mime_type};base64,{image_b64}"}},
#         ]
#         model = schema_model or watermark_model()
#         resp = await client.chat.completions.create(
#             model=self.served,
#             temperature=VERDICT_TEMPERATURE,
#             messages=[{"role": "user", "content": content}],
#             response_format={"type": "json_schema", "json_schema": {"name": "extraction", "schema": flat_schema(model)}},
#         )
#         raw = resp.choices[0].message.content or "{}"
#         u = resp.usage
#         tu = {"promptTokens": u.prompt_tokens, "outputTokens": u.completion_tokens} if u else None
#         return to_extraction(parse_json(model, raw), tu)

In [ ]:
# PRICING: USD per 1 JUTA token (input/output) untuk estimasi biaya per analisis (main.py).
#   - Kimi K2.5 & Nova Pro: harga AWS Bedrock In-Region JAKARTA (ap-southeast-3) Standard/On-Demand tier
#     (BUKAN US East). Kimi Jakarta $0.72/$3.60; Nova Pro Jakarta on-demand $0.84/$3.36 (turunan dari
#     Priority $1.47/1.75 = Flex $0.42/0.5). Verifikasi di halaman pricing AWS (pilih region Jakarta).
#   - Qwen self-host: TIDAK ada biaya per-token (sewa GPU per-jam) → pricing None → tampil "self-host".
PRICING = {
    "kimi-k2-5-bedrock": {"inUsdPerM": 0.72, "outUsdPerM": 3.6},
    "nova-pro-bedrock": {"inUsdPerM": 0.84, "outUsdPerM": 3.36},
    "qwen3-vl-vastai": None,
}

MODEL = {"kimi-k2-5-bedrock": KimiBedrockProvider("Kimi K2.5 (AWS Bedrock)"),
         "nova-pro-bedrock": NovaBedrockProvider("Nova Pro (AWS Bedrock)"),
         # "qwen3-vl-vastai": QwenVllmProvider("Qwen3-VL 32B (Vast.ai self-host)"),
         }


def _cost_usd(model_id: str, token_usage: dict | None) -> float | None:
    """Estimasi biaya 1 analisis = token × harga/1jt. None bila self-host (no pricing) / token tak ada."""
    p = PRICING.get(model_id)
    if not p or not token_usage:
        return None
    pt = token_usage.get("promptTokens") or 0
    ot = token_usage.get("outputTokens") or 0
    return round(pt / 1e6 * p["inUsdPerM"] + ot / 1e6 * p["outUsdPerM"], 6)


def _combine_usage(*usages: dict | None) -> dict | None:
    """Gabung beberapa token_usage (call fakta + call ekstraksi)."""
    us = [u for u in usages if u]
    if not us:
        return None
    return {"promptTokens": sum(u.get("promptTokens") or 0 for u in us),
            "outputTokens": sum(u.get("outputTokens") or 0 for u in us)}

## 7 · Evaluation

One photo = one test case; its folder is `groundTruth` (`positive` = pass, `negative` = finding). With no verdict yet, an Exp is read through its facts (`attribute_breakdown`). Settings are parameters, so no Exp changes anything global.

| Function | Does | Model call |
|---|---|---|
| `run_exp(photos=..., prompt=..., ...)` | facts + threshold check for the photos given, nothing else (no comparison, no rows from other Exps) | yes |
| `reevaluate(rows, rules=..., ...)` | the threshold check again on stored facts | no |
| `dump_prompt(department, sub_category, prompt=..., ...)` | prints exactly what the model receives | no |
| `run_layer2(rows, layer2=..., ...)` | layer 2 (8.10, off until used): a second look at chosen attributes on top of `run_exp` rows | yes, rechecked rows only |

Settings are keyword parameters of all of them (typing `run_exp(` lists them): write them directly, `run_exp(prompt=P, models=["nova-pro-bedrock"])`, or unpack an Exp dict, `run_exp(**exp3_1)`. Not given = `BASELINE`:

| Setting | Point | Default | Meaning |
|---|---|---|---|
| `models` | 8.8 | `MODELS` | models to run |
| `rules` | 8.2, 8.3, 8.6 | `{}` | `"dept/sub"` → rules (section 8 format) instead of the catalog's |
| `prompt`, `render` | 8.4 | `PROMPTS["facts"]`, `render_rules` | template text; rules → `{rules}` text |
| `schema` | 8.5 | `build_output_model` | attributes → output model |
| `references` | 8.7 | `{}` | `"dept/sub"` → `{"positive": [key], "negative": [key]}`, tried first |
| `max_refs`, `exclude_self`, `exact_references` | 8.7 | `5`, `True`, `False` | refs per role; never the photo itself; use the set as given |
| `extract`, `extract_prompt`, `extract_schema` | 8.9 | `True`, `PROMPTS["extract"]`, `watermark_model()` | watermark call (parallel), its text, its model |

- **Photos** (`photos=`): left out = every photo in the section 1 scope with rules; `photos_of(dept, main=, sub=)` (any sub-category); keys from the folders (`["images/…/a.jpg"]`); or `new_photo(key, dept, sub)` (a photo outside them).
- **References**: the set (`references`, else the catalog) first, then the folder by name, `max_refs` per role, the judged photo skipped and replaced by the next. `exact_references=True` uses the set as given. At most `MAX_IMAGES` per request (Bedrock 20).
- **Row** per photo × model: `facts` (`None` when off-schema, text in `rawText`), `thresholdChecks` / `thresholdFailed`, `groundTruth`, tokens, cost, references, prompt, watermark columns. Which Exp is best is your call; `compare_attributes` sets Exps side by side.

In [ ]:
BASELINE = dict(
    models=None, rules={},                                                              # 8.8, 8.2 / 8.3 / 8.6
    prompt=None, render=None, schema=None,                                              # 8.4, 8.5
    references={}, max_refs=5, exclude_self=True, exact_references=False,               # 8.7
    extract=True, extract_prompt=None, extract_schema=None,                             # 8.9
)


def settings(**changes) -> dict:
    """BASELINE + perubahan satu Exp; None = nilai BASELINE. Nama yang tidak dikenal langsung ditolak, jadi salah ketik
    ketahuan."""
    unknown = sorted(set(changes) - set(BASELINE))
    if unknown:
        raise TypeError(f"setting tidak dikenal: {unknown}; yang tersedia: {sorted(BASELINE)}")
    return {**BASELINE, **{k: v for k, v in changes.items() if v is not None}}


def discover(department=None):
    """Tiap foto di MANIFEST -> 1 soal: {departmentId, mainCategoryId, mainCategoryName, subCategoryId, role,
    groundTruth, relPath, image}. Semua sub-category, dengan atau tanpa rule."""
    tasks = []
    for dept in _load()["departments"]:
        if department and dept["id"] != department.lower():
            continue
        for mc in dept["mainCategories"]:
            for c in mc["subCategories"]:
                for role, gt in (("positive", "pass"), ("negative", "fail")):
                    for rel in (c.get(role) or []):
                        tasks.append({"departmentId": dept["id"], "mainCategoryId": mc["id"], "mainCategoryName": mc["name"],
                                      "subCategoryId": c["id"], "role": role,
                                      "groundTruth": gt, "relPath": rel, "image": os.path.basename(rel)})
    return tasks


def photos_of(department=None, main=None, sub=None) -> list[dict]:
    """Foto di MANIFEST untuk satu department / main category (id) / sub-category (id), dengan atau tanpa rule; apa pun
    cakupan bagian 1."""
    return [t for t in discover(department) if main in (None, t["mainCategoryId"]) and sub in (None, t["subCategoryId"])]


def new_photo(key: str, department: str, sub_category: str, ground_truth: str = "") -> dict:
    """Foto mana pun di bucket (juga di luar MANIFEST), dinilai sebagai foto sub-category itu."""
    mc, c = find_category(department, sub_category)
    if not c:
        raise ValueError(f"sub-category {department}/{sub_category} tidak ada di catalog")
    return {"departmentId": department, "mainCategoryId": mc["id"], "mainCategoryName": mc["name"],
            "subCategoryId": sub_category, "role": "", "groundTruth": ground_truth,
            "relPath": key, "image": os.path.basename(key)}


def _as_task(p) -> dict:
    if isinstance(p, dict):
        return p
    found = [t for t in discover() if t["relPath"] == p]
    if not found:
        raise ValueError(f"{p} tidak ada di MANIFEST; pakai new_photo(key, department, sub_category)")
    return found[0]


def _ref_candidates(department, sub_category, skip, s):
    """Kandidat referensi per role (urut) dan batasnya. Set pilihan (`references`, else katalog) dulu; rolling: lalu
    foto lain di folder, batas max_refs; exact: set saja, tanpa batas. exclude_self: foto yang dinilai dibuang."""
    _, c = find_category(department, sub_category)
    if not c:
        return {}, None
    chosen = s["references"].get(f"{department}/{sub_category}") or c.get("references") or {}
    candidates = {}
    for role in ("positive", "negative"):
        keys = list(chosen.get(role) or [])
        if not s["exact_references"]:
            keys = list(dict.fromkeys([*keys, *(c.get(role) or [])]))
        candidates[role] = [k for k in keys if not (s["exclude_self"] and k == skip)]
    return candidates, (None if s["exact_references"] else s["max_refs"])


def pick_references(department, sub_category, skip=None, s=None) -> list[RefImage]:
    """Foto referensi yang dikirim untuk foto `skip`; gambar yang gagal dibaca diganti kandidat berikutnya."""
    candidates, limit = _ref_candidates(department, sub_category, skip, s or BASELINE)
    out = []
    for role, keys in candidates.items():
        picked = 0
        for key in keys:
            if limit is not None and picked >= limit:
                break
            b64 = _encode(key)
            if b64:
                out.append(RefImage(base64=b64, mime_type="image/jpeg", role=role, label=os.path.basename(key), url=key))
                picked += 1
    return out

In [ ]:
def _threshold_fields(checks: list) -> dict:
    return {"thresholdChecks": checks, "thresholdFailed": [c["name"] for c in checks if not c["ok"]]}


async def run_one(task: dict, model_id: str, prov, s: dict) -> dict:
    """Satu foto x satu model: rule -> foto 1280 px + referensi -> fakta per atribut (model) -> cek threshold (kode,
    bagian 5), plus ekstraksi watermark paralel. s = settings(). Error apa pun jadi baris dengan kolom error, supaya
    satu foto gagal tidak menghentikan batch."""
    t0 = time.time()
    dept, sub = task["departmentId"], task["subCategoryId"]
    base = {k: task[k] for k in ("departmentId", "mainCategoryId", "mainCategoryName", "subCategoryId", "role",
                                 "groundTruth", "image", "relPath")}
    base.update(model=model_id, modelLabel=prov.label)
    try:
        _, c = find_category(dept, sub)
        rules = active_rules(dept, sub, s["rules"])
        if not rules:
            raise ValueError(f"{dept}/{sub} belum punya rule: isi lewat rules= (change point 8.2)")
        b64, mime = encode_photo(task["relPath"])
        refs = pick_references(dept, sub, skip=task["relPath"], s=s)
        limit = MAX_IMAGES.get(model_id, 20)
        if len(refs) + 1 > limit:
            raise ValueError(f"{len(refs)} reference + 1 foto melebihi limit {limit} image per request untuk {model_id}")
        prompt_text = render_prompt(s["prompt"] or PROMPTS["facts"], c, rules, s["render"])
        facts_call = prov.analyze(b64, mime, refs, prompt_text, (s["schema"] or build_output_model)(attributes_of(rules)))
        if s["extract"] and hasattr(prov, "extract"):
            # Fakta + ekstraksi watermark = 2 call PARALEL, seperti /api/analyze
            f, ext = await asyncio.gather(facts_call, prov.extract(b64, mime, prompt_text=s["extract_prompt"],
                                                                   schema_model=s["extract_schema"]))
        else:
            f, ext = await facts_call, Extraction()
        try:
            fields, error = _threshold_fields(check_thresholds(rules, f.facts)), ""
        except Exception as e:   # fakta tetap disimpan (sudah dibayar): bisa dicek ulang dengan reevaluate
            fields, error = {"thresholdChecks": [], "thresholdFailed": []}, f"threshold: {type(e).__name__}: {e}"
        tu = _combine_usage(f.token_usage, ext.token_usage) or {}
        return {**base, **fields, "facts": f.facts,
                "rawText": "" if f.facts else f.raw_text,
                "promptTokens": tu.get("promptTokens"), "outputTokens": tu.get("outputTokens"),
                "costUsd": _cost_usd(model_id, tu), "refCount": len(refs), "references": [r.url for r in refs],
                "latencyMs": int((time.time() - t0) * 1000), "prompt": prompt_text, "error": error,
                "capturedAt": ext.captured_at, "latitudeText": ext.latitude_text, "longitudeText": ext.longitude_text,
                "latitude": ext.latitude, "longitude": ext.longitude, "address": ext.address}
    except Exception as e:
        return {**base, "thresholdChecks": [], "thresholdFailed": [], "facts": None, "promptTokens": None, "outputTokens": None, "costUsd": None, "refCount": None,
                "latencyMs": int((time.time() - t0) * 1000), "error": f"{type(e).__name__}: {e}"}


CONCURRENCY = {"kimi-k2-5-bedrock": 8, "nova-pro-bedrock": 2, "qwen3-vl-vastai": 6}   # per pod; batas laju Bedrock dipakai bersama semua pod
MAX_IMAGES = {"kimi-k2-5-bedrock": 20, "nova-pro-bedrock": 20, "qwen3-vl-vastai": 15}   # gambar per request (referensi + 1 foto)


async def run_batch(pairs: list, s: dict) -> list:
    """Jalankan semua pasangan (foto, model) bersamaan; satu semaphore per model (CONCURRENCY): begitu satu foto
    selesai, foto berikutnya masuk."""
    sems = {m: asyncio.Semaphore(CONCURRENCY.get(m, 4)) for m in {m for _, m in pairs}}

    async def limited(task, model_id):
        async with sems[model_id]:
            return await run_one(task, model_id, MODEL[model_id], s)

    rows = await asyncio.gather(*(limited(t, m) for t, m in pairs))
    print(f"{len(rows)} call done, {sum(1 for r in rows if r['error'])} error")
    return list(rows)


def _check_all(pairs, s: dict) -> None:
    """check_rules untuk tiap sub-category yang akan dinilai (yang punya rule), sebelum model dipanggil."""
    for dept, sub in sorted(set(pairs)):
        rules = active_rules(dept, sub, s["rules"])
        if rules:
            check_rules(dept, sub, rules)


async def run_exp(
    photos: list | None = None,
    *,
    models: list[str] | None = None,
    rules: dict[str, list[dict]] | None = None,
    prompt: str | None = None,
    render: Callable | None = None,
    schema: Callable | None = None,
    references: dict[str, dict[str, list[str]]] | None = None,
    max_refs: int | None = None,
    exclude_self: bool | None = None,
    exact_references: bool | None = None,
    extract: bool | None = None,
    extract_prompt: str | None = None,
    extract_schema: type[BaseModel] | None = None,
) -> list:
    """Fakta + cek threshold oleh kode, untuk foto yang diminta saja (tidak dibandingkan atau diisi dari Exp lain).
    Setelan ditulis langsung, run_exp(prompt=P, models=[...]), atau dari dict Exp, run_exp(**exp3_1). Setelan yang
    tidak diisi (None) = nilai BASELINE (bagian 7). Angka (8.x) = change point di bagian 8.

    Args:
        photos: foto yang dinilai. None = semua foto di cakupan bagian 1 yang sub-category-nya punya rule;
            photos_of("dept_a", sub="fr-a-02") = satu sub-category; ["images/.../a.jpg"] = foto tertentu di folder;
            [new_photo(key, "dept_a", "fr-a-02")] = foto di luar folder.
        models: (8.8) id model yang dijalankan, mis. ["nova-pro-bedrock"]. None = MODELS (bagian 1).
        rules: (8.2 / 8.3 / 8.6) {"dept/sub": [rule, ...]} pengganti rule katalog untuk sub-category itu, format
            bagian 8. Sub-category lain tetap rule katalog. None = rule katalog.
        prompt: (8.4) teks template prompt fakta, dengan {sub_category_id} {requirement} {rules}. None = PROMPTS["facts"].
        render: (8.4) fungsi rules -> teks {rules} di prompt. None = render_rules.
        schema: (8.5) fungsi atribut -> model output Pydantic. None = build_output_model.
        references: (8.7) {"dept/sub": {"positive": [key], "negative": [key]}} set referensi pilihan, dicoba paling
            dulu. None = references di katalog.
        max_refs: (8.7) jumlah referensi per role (positive / negative). None = 5.
        exclude_self: (8.7) foto yang dinilai tidak dipakai jadi referensinya sendiri, juga saat exact_references=True
            (set yang dikunci). False hanya untuk uji coba satu foto. None = True.
        exact_references: (8.7) True = set referensi dipakai apa adanya: tanpa tambahan foto folder, tanpa batas
            max_refs. None = False (set dulu, lalu foto folder urut nama).
        extract: (8.9) jalankan ekstraksi watermark (tanggal, lat / long, alamat) sebagai call kedua. None = True.
        extract_prompt: (8.9) teks prompt ekstraksi watermark. None = PROMPTS["extract"].
        extract_schema: (8.9) model output ekstraksi. None = watermark_model().

    Returns:
        Satu baris (dict) per foto x model: facts, thresholdChecks, thresholdFailed, groundTruth, token, biaya,
        referensi, prompt, kolom ekstraksi. Baca dengan run_summary, attribute_breakdown, facts_frame.
    """
    s = settings(**{k: v for k, v in locals().items() if k in BASELINE})   # parameter setelan di atas
    models = s["models"] or MODELS
    tasks = [_as_task(p) for p in (_default_photos() if photos is None else photos)]
    _check_all(((t["departmentId"], t["subCategoryId"]) for t in tasks), s)
    return await run_batch([(t, m) for t in tasks for m in models], s)

`reevaluate` (free) checks thresholds again on stored facts; any other rule change needs `run_exp`. Pass the rules to check, `reevaluate(hasil["example4.1"], rules={"dept_c/fr-c-04": R})`; it re-checks **every** row given, and a sub-category without `rules=` falls back to the catalog. A threshold on an attribute that was not asked makes that row an error. It lists the photos whose `thresholdFailed` changed.

In [ ]:
def reevaluate(
    rows: list,
    *,
    models: list[str] | None = None,
    rules: dict[str, list[dict]] | None = None,
    prompt: str | None = None,
    render: Callable | None = None,
    schema: Callable | None = None,
    references: dict[str, dict[str, list[str]]] | None = None,
    max_refs: int | None = None,
    exclude_self: bool | None = None,
    exact_references: bool | None = None,
    extract: bool | None = None,
    extract_prompt: str | None = None,
    extract_schema: type[BaseModel] | None = None,
) -> list:
    """Cek threshold diulang pada fakta yang sudah ada di `rows`, tanpa memanggil model (gratis, change point 8.6).
    Kirim rule dengan threshold baru, mis. reevaluate(hasil["example4.1"], rules={"dept_c/fr-c-04": RULES_RSL}); sub-category
    tanpa rules= memakai rule katalog. Baris yang faktanya tidak memuat atribut rule baru jadi error: jalankan run_exp.

    Args:
        rows: hasil run_exp sebelumnya, mis. hasil["example4.1"].
        rules: (8.6) {"dept/sub": [rule, ...]} dengan threshold baru. None = rule katalog. Hanya setelan ini yang
            dipakai di sini; setelan lain diterima supaya **expN_k bisa dikirim utuh, tapi tidak berpengaruh.

    Returns:
        Baris yang sama dengan thresholdChecks / thresholdFailed dihitung ulang; foto yang berubah dicetak.
    """
    s = settings(**{k: v for k, v in locals().items() if k in BASELINE})   # parameter setelan di atas
    _check_all(((r["departmentId"], r["subCategoryId"]) for r in rows if r.get("facts") is not None), s)
    out = []
    for r in rows:
        if r.get("facts") is None:   # gagal sebelum fakta terbaca (tanpa rule, call gagal, output tidak valid)
            out.append(r)
            continue
        rules = active_rules(r["departmentId"], r["subCategoryId"], s["rules"])
        missing = sorted(set(attributes_of(rules)) - set(r["facts"]))
        if not rules or missing:
            why = f"atribut belum ditanya ke model: {missing}" if missing else "sub-category tanpa rule"
            out.append({**r, "thresholdChecks": [], "thresholdFailed": [], "error": f"{why}; pakai run_exp"})
            continue
        out.append({**r, **_threshold_fields(check_thresholds(rules, r["facts"])), "error": ""})
    changed = [{"subCategoryId": b["subCategoryId"], "image": b["image"], "model": b["model"],
                "groundTruth": b["groundTruth"], "sebelum": a.get("thresholdFailed"), "sesudah": b.get("thresholdFailed"),
                "error": b["error"]}
               for a, b in zip(rows, out) if a.get("thresholdFailed") != b.get("thresholdFailed") or b["error"]]
    print(f"{len(out)} row dicek ulang tanpa call model, {sum(1 for r in out if r['error'])} error, "
          f"thresholdFailed berubah di {len(changed)} row")
    if changed:
        display(pd.DataFrame(changed))
    return out

**Layer 2** (change point 8.10, **off until used**): a second look at chosen attributes on top of `run_exp` rows, so layer 1 does not run again. Per sub-category, `recheck` names the attributes and the layer-1 answer (`is_true`) to check again, e.g. `{"pondasi_beton_terlihat": False}` (or `"all"`); only photos with that answer get a second call, asking those attributes only. The model does not see the first answer. Template: the sub-category's `"prompt"`, else `PROMPTS["layer2"]`. The answer replaces those facts (`factsLayer1` keeps the first, `layer2Changed` names what changed) and thresholds are checked again. Settings: `layer2=` (the Exp), else a `layer2` field on the sub-category in the catalog (none yet). Worked example: section 9, Exp 4 (commented out).

In [ ]:
def _layer2_config(department: str, sub_category: str, layer2: dict | None) -> dict:
    """Setelan layer 2 satu sub-category: `layer2` (setelan Exp) bila ada untuk sub-category ini, kalau tidak field
    `layer2` di katalog (belum ada yang mengisi). {} = layer 2 mati."""
    key = f"{department}/{sub_category}"
    if layer2 is not None and key in layer2:
        return layer2[key] or {}
    _, c = find_category(department, sub_category)
    return (c or {}).get("layer2") or {}


def _layer2_targets(row: dict, recheck) -> list[str]:
    """Atribut baris ini yang dicek ulang: "all" = semua; {nama: jawaban} = atribut yang is_true layer 1-nya = jawaban."""
    facts = row.get("facts") or {}
    if recheck == "all":
        return [k for k, f in facts.items() if isinstance(f, dict)]
    return [k for k, want in (recheck or {}).items() if isinstance(facts.get(k), dict) and facts[k].get("is_true") == want]


def _only_attributes(rules: list, names: list[str]) -> list:
    """Salinan rule yang hanya berisi atribut `names`; rule yang tidak punya atribut tersisa dibuang."""
    out = []
    for r in copy.deepcopy(rules):
        r["attributes"] = [a for a in r["attributes"] if a["name"] in names]
        if r["attributes"]:
            out.append(r)
    return out


async def run_layer2(
    rows: list,
    *,
    layer2: dict | None = None,
    models: list[str] | None = None,
    rules: dict[str, list[dict]] | None = None,
    prompt: str | None = None,
    render: Callable | None = None,
    schema: Callable | None = None,
    references: dict[str, dict[str, list[str]]] | None = None,
    max_refs: int | None = None,
    exclude_self: bool | None = None,
    exact_references: bool | None = None,
    extract: bool | None = None,
    extract_prompt: str | None = None,
    extract_schema: type[BaseModel] | None = None,
) -> list:
    """Layer 2 (change point 8.10): cek ulang atribut tertentu di atas hasil run_exp, tanpa menjalankan layer 1 lagi.
    Hanya foto yang jawaban layer 1-nya cocok dengan `recheck` yang dikirim ulang, dan hanya atribut itu yang ditanya.
    Model tidak melihat jawaban pertamanya.

    Args:
        rows: hasil run_exp, mis. hasil["example1.1"].
        layer2: {"dept/sub": {"recheck": {nama_atribut: jawaban is_true yang dicek ulang} atau "all",
            "prompt": teks template (opsional; default PROMPTS["layer2"])}}. Sub-category yang tidak disebut: field
            `layer2` di katalog; tidak ada = tidak dicek ulang.
        rules, references, max_refs, exclude_self, exact_references, prompt, render, schema: sama dengan run_exp;
            kirim setelan yang sama dengan run_exp baris itu supaya rule dan referensinya sama. Template layer 2
            datang dari `layer2`, bukan `prompt`. models tidak dipakai (model = model baris itu); ekstraksi selalu mati.

    Returns:
        Baris yang sama. Yang dicek ulang: facts = jawaban layer 2 untuk atribut itu, factsLayer1 = jawaban pertama,
        layer2Changed = atribut yang jawabannya berubah, threshold dicek ulang, biaya layer 2 ditambahkan ke costUsd
        (layer2CostUsd). Gagal: baris layer 1 tetap, alasannya di layer2Error.
    """
    s = settings(**{k: v for k, v in locals().items() if k in BASELINE})   # parameter setelan di atas
    groups = defaultdict(list)   # (dept, sub, atribut, template) -> index baris
    for i, r in enumerate(rows):
        if r.get("error") or not r.get("facts"):
            continue
        cfg = _layer2_config(r["departmentId"], r["subCategoryId"], layer2)
        targets = _layer2_targets(r, cfg.get("recheck"))
        if targets:
            groups[(r["departmentId"], r["subCategoryId"], tuple(targets), cfg.get("prompt") or PROMPTS["layer2"])].append(i)
    out = list(rows)
    for (dept, sub, targets, template), idx in groups.items():
        rules = active_rules(dept, sub, s["rules"])
        s2 = {**s, "rules": {**s["rules"], f"{dept}/{sub}": _only_attributes(rules, list(targets))},
              "prompt": template, "extract": False}
        second = await run_batch([(rows[i], rows[i]["model"]) for i in idx], s2)
        for i, r2 in zip(idx, second):
            r = rows[i]
            if r2["error"] or not r2.get("facts"):
                out[i] = {**r, "layer2Error": r2["error"] or "output layer 2 tidak sesuai schema"}
                continue
            new = {k: r2["facts"][k] for k in targets if k in r2["facts"]}
            facts = {**r["facts"], **new}
            changed = [k for k, f in new.items()
                       if (f.get("is_true"), f.get("value")) != (r["facts"][k].get("is_true"), r["facts"][k].get("value"))]
            try:
                fields, error = _threshold_fields(check_thresholds(rules, facts)), r["error"]
            except Exception as e:
                fields, error = {"thresholdChecks": [], "thresholdFailed": []}, f"threshold: {type(e).__name__}: {e}"
            out[i] = {**r, **fields, "facts": facts, "factsLayer1": r["facts"], "layer2Changed": changed,
                      "layer2Error": "", "error": error, "layer2CostUsd": r2.get("costUsd"),
                      "costUsd": (r.get("costUsd") or 0) + (r2.get("costUsd") or 0)}
    print(f"layer 2: {sum(len(v) for v in groups.values())} row dicek ulang, "
          f"jawaban berubah di {sum(1 for r in out if r.get('layer2Changed'))} row")
    return out

Reading results:
- `run_summary`: rows, errors, off-schema answers, threshold misses and cost per model (`by=PER_SUB`: per sub-category).
- `attribute_breakdown`, the main measure: per attribute, % true on positive and negative photos and `selisih` (the gap, in points). Large = tells the folders apart (negative for a defect like `retak`); near 0 = does not, or is misread.
- `compare_attributes`: several Exps side by side. `facts_frame`: one row per photo × model × attribute, the ground truth (photo folder) next to the AI answer (`is_true`, `value`, threshold, `evidence`); filter with `sub_category=`, `attribute=`, `model=`.
- `attribute_table`: the AI answers as a structured table: one row per photo × attribute (rule order) with `actual` (photo folder), grouped columns `is_true` / `value` / `evidence` per model, so Kimi and Nova sit side by side. `is_true` True means the attribute's statement holds, not "pass" (`retak` True = a crack shows). `watermark_table`: the watermark extraction per photo × model. Both filter with `sub_category=`, `model=`.

In [ ]:
def run_summary(rows: list, by=None) -> pd.DataFrame:
    """Per model, atau per kelompok + model bila `by` diisi (mis. by=PER_SUB): jumlah baris, error, output tidak sesuai
    schema (fakta None), foto positif / negatif dengan fakta, foto dengan threshold tidak terpenuhi, dan biaya."""
    keys = [by] if isinstance(by, str) else list(by or [])
    agg = defaultdict(lambda: {"n": 0, "error": 0, "invalid": 0, "pos": 0, "neg": 0, "threshold": 0, "usd": 0.0})
    for r in rows:
        a = agg[tuple(r.get(k) for k in keys) + (r["model"],)]
        a["n"] += 1
        if r.get("error"):
            a["error"] += 1
            continue
        a["usd"] += r.get("costUsd") or 0
        if r.get("facts") is None:
            a["invalid"] += 1
            continue
        a["pos"] += r.get("groundTruth") == "pass"
        a["neg"] += r.get("groundTruth") == "fail"
        a["threshold"] += bool(r.get("thresholdFailed"))
    columns = [*keys, "Model", "Total", "Error", "Output tidak valid", "n(pos)", "n(neg)", "Threshold gagal", "Cost USD"]
    return pd.DataFrame([[*key, a["n"], a["error"], a["invalid"], a["pos"], a["neg"], a["threshold"], round(a["usd"], 4)]
                         for key, a in agg.items()], columns=columns)


def _pct(part: int, whole: int) -> float | None:
    return round(part / whole * 100, 1) if whole else None


BREAKDOWN_COLUMNS = ["subCategoryId", "attribute", "Model", "n(pos)", "true di pos", "% pos", "n(neg)", "true di neg",
                     "% neg", "selisih", "threshold gagal"]


def attribute_breakdown(rows: list) -> pd.DataFrame:
    """Per sub-category x atribut x model: seberapa sering atribut bernilai true (is_true) di foto positif vs negatif,
    selisihnya (% pos - % neg), dan berapa kali threshold-nya tidak terpenuhi. Selisih mendekati 0 = atribut tidak
    membedakan foto positif dan negatif."""
    agg = defaultdict(lambda: {"pos": 0, "pos_true": 0, "neg": 0, "neg_true": 0, "threshold_fail": 0})
    for r in rows:
        if r.get("error") or not r.get("facts") or r.get("groundTruth") not in ("pass", "fail"):
            continue
        side = "pos" if r["groundTruth"] == "pass" else "neg"
        failed = set(r.get("thresholdFailed") or [])
        for key, f in r["facts"].items():
            if not isinstance(f, dict):
                continue                                   # field tambahan dari schema Exp (mis. scene)
            a = agg[(r["subCategoryId"], key, r["model"])]
            a[side] += 1
            a[f"{side}_true"] += bool(f.get("is_true"))
            a["threshold_fail"] += key in failed
    out = []
    for k, a in sorted(agg.items()):
        pos, neg = _pct(a["pos_true"], a["pos"]), _pct(a["neg_true"], a["neg"])
        out.append([*k, a["pos"], a["pos_true"], pos, a["neg"], a["neg_true"], neg,
                    round(pos - neg, 1) if pos is not None and neg is not None else None, a["threshold_fail"]])
    return pd.DataFrame(out, columns=BREAKDOWN_COLUMNS)


def compare_attributes(results: dict, cols=("% pos", "% neg", "selisih")) -> pd.DataFrame:
    """attribute_breakdown beberapa Exp berdampingan: {nama: rows} -> satu blok kolom `cols` per nama."""
    return pd.concat({name: attribute_breakdown(rows).set_index(["subCategoryId", "attribute", "Model"])[list(cols)]
                      for name, rows in results.items()}, axis=1)


FACTS_COLUMNS = ["sub-category", "foto", "ground truth", "model", "atribut", "AI: is_true", "AI: value", "threshold",
                 "AI: evidence"]
GROUND_TRUTH = {"pass": "positive", "fail": "negative"}   # folder foto


def _threshold_text(check: dict) -> str:
    """Satu cek threshold untuk dibaca, mis. '>= -24 · gagal'."""
    cond = " dan ".join(f"{op} {x:g}" for op, x in check["threshold"].items())
    return f"{cond} · {'lolos' if check['ok'] else 'gagal'}"


def facts_frame(rows: list, department=None, sub_category=None, attribute=None, model=None) -> pd.DataFrame:
    """Jawaban AI per foto x model x atribut, di samping ground truth (folder foto: positive / negative).
    AI: is_true = pernyataan di description atribut benar untuk foto ini; AI: value = nilai yang terbaca (None untuk
    bool, atau bila tidak terbaca); threshold = syaratnya + hasil cek kode (hanya atribut number ber-threshold);
    AI: evidence = yang dilihat model. Filter: department, sub_category, attribute (nama atribut), model (id model)."""
    out = []
    for r in rows:
        if r.get("error") or not r.get("facts") or department not in (None, r["departmentId"]) \
                or sub_category not in (None, r["subCategoryId"]) or model not in (None, r["model"]):
            continue
        checks = {c["name"]: c for c in r.get("thresholdChecks") or []}
        for key, f in r["facts"].items():
            if attribute not in (None, key):
                continue
            if not isinstance(f, dict):
                f = {"value": f}                               # field tambahan dari schema Exp (mis. scene)
            out.append([r["subCategoryId"], r["image"], GROUND_TRUTH.get(r.get("groundTruth"), "-"), r["model"], key,
                        f.get("is_true"), f.get("value"), _threshold_text(checks[key]) if key in checks else "",
                        f.get("evidence")])
    df = pd.DataFrame(out, columns=FACTS_COLUMNS)
    return df.sort_values(["sub-category", "atribut", "model", "ground truth", "foto"], kind="stable").reset_index(drop=True)


WATERMARK_KEYS = ("capturedAt", "latitudeText", "longitudeText", "latitude", "longitude", "address")   # kolom ekstraksi


def attribute_table(rows: list, sub_category=None, model=None) -> pd.DataFrame:
    """Jawaban AI sebagai tabel terstruktur: satu baris per sub-category x foto x atribut (urutan rule), dengan actual =
    label asli foto (folder positive / negative); kolom berkelompok per field x model: is_true, value (hanya bila ada
    atribut text / number / json), evidence. Kimi dan Nova bersebelahan per atribut. is_true True = pernyataan di
    description atribut benar untuk foto itu, BUKAN berarti lolos: atribut cacat (mis. retak) True = cacatnya terlihat.
    Filter: sub_category, model (id model). Data per baris (satu kolom per field): facts_frame."""
    ff = facts_frame(rows, sub_category=sub_category, model=model).rename(columns={
        "ground truth": "actual", "AI: is_true": "is_true", "AI: value": "value", "AI: evidence": "evidence"})
    order = list(dict.fromkeys(k for r in rows if r.get("facts") for k, f in r["facts"].items() if isinstance(f, dict)))
    ff = ff[ff["atribut"].isin(order)].copy()                            # field tambahan schema Exp (mis. scene) tidak ikut
    ff["atribut"] = pd.Categorical(ff["atribut"], categories=order)      # urutan rule, bukan abjad
    return (ff.pivot(index=["sub-category", "foto", "actual", "atribut"], columns="model",
                     values=["is_true", "value", "evidence"])
            .dropna(axis=1, how="all"))                                   # value kosong (semua bool) -> dibuang


def watermark_table(rows: list, sub_category=None, model=None) -> pd.DataFrame:
    """Hasil ekstraksi watermark per foto x model: teks lintang / bujur apa adanya, hasil desimalnya, tanggal, alamat;
    di samping actual (folder foto). Baris error dilewati. Filter: sub_category, model (id model)."""
    out = [{"sub-category": r["subCategoryId"], "foto": r["image"], "model": r["model"],
            "actual": GROUND_TRUTH.get(r.get("groundTruth"), "-"), **{k: r.get(k) for k in WATERMARK_KEYS}}
           for r in rows if not r.get("error") and sub_category in (None, r["subCategoryId"])
           and model in (None, r["model"])]
    return pd.DataFrame(out, columns=["sub-category", "foto", "model", "actual", *WATERMARK_KEYS])

**Ground truth** (`notebook/ground_truth/<dept>/<sub>__<model>.xlsx`, git-ignored in this public copy): one row per photo × attribute: `photo`, `folder`, `rule` (for reading), `attribute`, `actual` = `positive` / `negative`: whether the attribute passes on that photo, after `expected` (`retak` with `expected: false` on a cracked pole = `negative`). The file is the ground truth as it is: fix `actual` directly. Two steps:
1. `save_ground_truth_draft(rows, model=..., rules=...)`: one model's answers (e.g. `model="kimi-k2-5-bedrock"`) become the ground truth file; `rules` the same as the run. An existing file is never changed: only new photo × attribute rows are added.
2. Fix `actual` in Excel where it is wrong, save as `.xlsx`, commit. Then `compare_ground_truth(rows, rules=...)`: the AI's positive / negative per rule × attribute × model against `actual` (% correct, wrong positives / negatives, the photos it got wrong), closed per model by an "semua atribut" row: all attributes together. Shown first, per department × model: prediction × actual over all photo × attribute pairs (total attribute). `save="exp1.3_vs_gt"` (any name) also writes `hasil-experiment/<dept>/<name>.xlsx`: sheet `summary` (photos, attributes, rules per sub-category and the prediction × actual table with accuracy per model) and sheet `ground truth vs predicted` (the ground truth rows plus a `predicted_<model>` column per model); commit it so others can read the result.

An old run, from its CSV, without calling the models again: `compare_ground_truth(exporter.load("exp1.3"), rules=...)`.

In [ ]:
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

GROUND_TRUTH_DIR = REPO / "notebook" / "ground_truth"   # <dept>/<sub>.xlsx: satu baris per foto x atribut
GT_COLUMNS = ["photo", "folder", "rule", "attribute", "actual"]
VALUE_COLUMN = "value"   # opsional, setelah actual: angka yang tertera di foto untuk atribut number
GT_WIDTHS = {"photo": 62, "folder": 10, "rule": 28, "attribute": 26, "actual": 10, "value": 12}
COMPARE_DIR = REPO / "notebook" / "hasil-experiment"   # compare_ground_truth(save="nama"): <dept>/<nama>.xlsx


def _ground_truth_path(department: str, sub_category: str, model: str) -> Path:
    return GROUND_TRUTH_DIR / department / f"{sub_category}__{model}.xlsx"


def _read_ground_truth(path: Path) -> pd.DataFrame:
    df = pd.read_excel(path, dtype=str).fillna("")
    missing = [c for c in GT_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name}: kolom {missing} tidak ada; kolom ground truth: {GT_COLUMNS}")
    return df


def _number(value) -> float | None:
    """Angka dari sel Excel atau jawaban AI ("-19,34" dan tanda minus "−" juga dibaca); None bila kosong / bukan angka."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return None if value != value else float(value)      # NaN = sel kosong
    text = str(value).strip().replace("−", "-")
    if "," in text and "." not in text:
        text = text.replace(",", ".")
    try:
        return float(text)
    except ValueError:
        return None


def _number_text(value: float) -> str:
    """-19.34 -> "-19.34", 1490.0 -> "1490"."""
    return format(value, ".10g")


def _read_number(spec: dict, fact: dict | None) -> float | None:
    """Angka yang dibaca AI untuk atribut number (is_true dan value berupa angka); selain itu None."""
    fact = fact or {}
    return _number(fact.get("value")) if spec.get("type") == "number" and fact.get("is_true") else None


def _natural_key(text: str) -> list:
    """Urutan seperti folder di Windows Explorer: angka dibaca sebagai angka (FULL(2) sebelum FULL(10)), huruf besar /
    kecil dianggap sama."""
    return [int(t) if i % 2 else t.casefold() for i, t in enumerate(re.split(r"(\d+)", text))]


def _sorted_by_photo(df: pd.DataFrame) -> pd.DataFrame:
    """Baris ground truth diurutkan per foto (_natural_key); urutan atribut di dalam satu foto tetap."""
    return df.iloc[sorted(range(len(df)), key=lambda i: _natural_key(df["photo"].iat[i]))].reset_index(drop=True)


def _write_ground_truth(df: pd.DataFrame, path: Path, sheet: str) -> None:
    """Simpan ke Excel: header tebal dan dibekukan, filter, lebar kolom. Format buatan user (warna dll.) tidak ikut."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if VALUE_COLUMN in df.columns:   # angka ditulis sebagai angka Excel; isi lain (kosong, teks) apa adanya
        df = df.assign(**{VALUE_COLUMN: [v if _number(v) is None else _number(v) for v in df[VALUE_COLUMN]]})
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet)
        ws = writer.sheets[sheet]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.font = Font(bold=True)
        for i, name in enumerate(df.columns, 1):
            ws.column_dimensions[get_column_letter(i)].width = GT_WIDTHS.get(name, 15)


def _load_ground_truth(department: str, sub_category: str, model: str) -> dict:
    path = _ground_truth_path(department, sub_category, model)
    if not path.is_file():
        return {}
    out = {}
    for n, row in enumerate(_read_ground_truth(path).to_dict("records"), 2):
        text = row["actual"].strip().lower()
        if text not in ("", "positive", "negative"):
            raise ValueError(f"{path.name} baris {n}: actual harus positive / negative / kosong, bukan {row['actual']!r}")
        cells = out.setdefault(row["photo"], {})
        if row["attribute"] in cells:
            raise ValueError(f"{path.name} baris {n}: {row['photo']} / {row['attribute']} sudah ada di baris lain")
        raw = str(row.get(VALUE_COLUMN, "")).strip()
        if raw and _number(raw) is None:
            raise ValueError(f"{path.name} baris {n}: value harus angka (angka yang tertera di foto), bukan {raw!r}")
        cells[row["attribute"]] = {"actual": None if text == "" else text == "positive", "rule": row["rule"],
                                   "value": _number(raw)}
    return out


def _rule_titles(rules: list) -> dict:
    """Atribut -> judul rule-nya; atribut di beberapa rule: judulnya digabung " / "."""
    titles = defaultdict(list)
    for r in rules:
        for a in r["attributes"]:
            titles[a["name"]].append(r["title"])
    return {name: " / ".join(t) for name, t in titles.items()}


def _passed(specs: dict, name: str, fact: dict | None) -> bool:
    """Atribut positive (lolos): yang terlihat di foto sama dengan expected (bawaan true, juga untuk atribut di luar rule
    dan notebook yang belum memakai expected). Terlihat: bool / text = is_true; json = minimal satu item; number = angka
    terbaca dan memenuhi threshold bila ada. Aturannya sama dengan attribute_passed (verdict, bagian 5), ditulis ulang di
    sini supaya sel ini bisa disalin apa adanya ke notebook tanpa verdict (DEPT-A / DEPT-B)."""
    spec, fact = specs.get(name, {}), fact or {}
    if spec.get("threshold"):
        seen = threshold_ok(spec, fact)
    elif spec.get("type") in ("json", "number"):
        seen = bool(fact.get("is_true")) and fact.get("value") not in (None, [])
    else:
        seen = bool(fact.get("is_true"))
    return seen == spec.get("expected", True)


def _misread(specs: dict, name: str, fact: dict | None, cell: dict) -> bool:
    """Atribut number yang ground truth-nya positive dengan kolom value terisi, dan AI membaca angka lain. Positive /
    negative tetap soal terlihat (bisa dinilai tanpa ground truth); angka yang salah dihitung salah di compare, karena
    threshold di decision layer memakai angka itu."""
    want = cell.get("value")
    if want is None or not cell.get("actual"):
        return False
    got = _read_number(specs.get(name, {}), fact)
    return got is not None and abs(got - want) > 1e-9


def save_ground_truth_draft(rows: list, model: str, rules: dict | None = None) -> None:
    """Langkah 1: hasil run_exp -> ground truth, satu Excel per sub-category, satu baris per foto x atribut. actual =
    positive / negative jawaban `model` (mis. "kimi-k2-5-bedrock") setelah expected. rules: setelan rules= yang sama
    dengan run_exp (None = rule katalog), untuk expected dan kolom rule. File yang sudah ada TIDAK diubah (hasil edit
    aman): hanya pasangan foto x atribut yang belum ada yang ditambahkan. Baris diurutkan per foto seperti folder
    (_natural_key). Atribut number: kolom value = angka yang dibaca `model`, untuk direview (isi angka yang tertera di
    foto). Di file yang sudah ada, value yang masih kosong ikut diisi bila actual positive (atau atribut ber-threshold);
    actual dan isi lain tetap."""
    answers = defaultdict(dict)   # (dept, sub) -> foto -> facts dari `model`
    folders = {}
    for r in rows:
        if r.get("error") or not r.get("facts") or r.get("model") != model:
            continue
        answers[(r["departmentId"], r["subCategoryId"])][r["relPath"]] = r["facts"]
        folders[r["relPath"]] = GROUND_TRUTH.get(r.get("groundTruth"), "")
    if not answers:
        raise ValueError(f"tidak ada jawaban {model!r} di rows; model yang ada: "
                         f"{sorted({r['model'] for r in rows if r.get('model')})}")
    for (dept, sub), photos in answers.items():
        path = _ground_truth_path(dept, sub, model)
        old = _read_ground_truth(path) if path.is_file() else pd.DataFrame(columns=GT_COLUMNS)
        known = set(zip(old["photo"], old["attribute"]))
        sub_rules = active_rules(dept, sub, rules)
        specs, rule_of = attributes_of(sub_rules), _rule_titles(sub_rules)
        numbers = {a for a, spec in specs.items() if spec.get("type") == "number"}

        def drafted(a: str, fact: dict) -> str:   # angka bacaan `model` untuk kolom value ("" bila bukan number / tidak terbaca)
            value = _read_number(specs.get(a, {}), fact)
            return "" if value is None else _number_text(value)

        new_rows = [{"photo": photo, "folder": folders[photo], "rule": rule_of.get(a, ""), "attribute": a,
                     "actual": "positive" if _passed(specs, a, fact) else "negative", VALUE_COLUMN: drafted(a, fact)}
                    for photo in sorted(photos, key=_natural_key) for a, fact in photos[photo].items()
                    if isinstance(fact, dict) and (photo, a) not in known]
        filled = 0
        if numbers and len(old):                   # file lama: isi value yang masih kosong, actual tidak disentuh
            if VALUE_COLUMN not in old.columns:
                old[VALUE_COLUMN] = ""
            for i, row in old.iterrows():
                fact = photos.get(row["photo"], {}).get(row["attribute"])
                wanted = row["actual"].strip().lower() == "positive" or specs.get(row["attribute"], {}).get("threshold")
                if row["attribute"] in numbers and not str(row[VALUE_COLUMN]).strip() and wanted \
                        and isinstance(fact, dict) and drafted(row["attribute"], fact):
                    old.at[i, VALUE_COLUMN] = drafted(row["attribute"], fact)
                    filled += 1
        if not new_rows and not filled:
            print(f"ground_truth/{dept}/{sub}__{model}.xlsx: tidak ada foto x atribut baru, file tidak diubah")
            continue
        value_column = [VALUE_COLUMN] if numbers or VALUE_COLUMN in old.columns else []
        columns = GT_COLUMNS + value_column + [c for c in old.columns if c not in GT_COLUMNS + value_column]
        merged = _sorted_by_photo(pd.concat([old, pd.DataFrame(new_rows)], ignore_index=True)
                                  .reindex(columns=columns).fillna(""))
        _write_ground_truth(merged, path, sub)
        print(f"ground_truth/{dept}/{sub}__{model}.xlsx: {len(new_rows)} baris baru dari {model}"
              + (f", value dari {model} diisi di {filled} sel yang masih kosong (actual tidak diubah)" if filled
                 else f", {len(old)} baris lama tidak diubah"))


def _confusion_tables(confusion: dict, scope: dict, rule_counts: dict, misreads: dict) -> list:
    """[(department, model, judul, tabel)]: prediksi AI (baris) x actual ground truth (kolom) dari semua foto x atribut
    (positive / negative setelah expected), plus TOTAL. Judul: asal total (foto x atribut, dari berapa rule, per
    sub-category) dan akurasi. Angka salah (misreads: AI dan GT positive, angka beda dengan kolom value) tetap di kotak
    POSITIVE x POSITIVE tapi tidak dihitung benar; jumlahnya ditulis di judul."""
    label, tables = {True: "POSITIVE", False: "NEGATIVE"}, []
    for dept, model in dict.fromkeys(k[:2] for k in confusion):
        parts = [f"{len(s['photos'])} foto x {len(s['attributes'])} atribut (dari {rule_counts[(dept, sub)]} rule)"
                 for (d, m, sub), s in scope.items() if (d, m) == (dept, model)]
        subs = [sub for d, m, sub in scope if (d, m) == (dept, model)]
        origin = parts[0] if len(parts) == 1 else " + ".join(f"{sub}: {p}" for sub, p in zip(subs, parts))
        table = pd.DataFrame([[confusion.get((dept, model, got, want), 0) for want in label] for got in label],
                             index=pd.Index(list(label.values()), name="Prediction"),
                             columns=pd.Index(list(label.values()), name="Actual"))
        table["TOTAL"] = table.sum(axis=1)
        table.loc["TOTAL"] = table.sum()
        misread = misreads.get((dept, model), 0)
        n = table.loc["TOTAL", "TOTAL"]
        ok = table.loc["POSITIVE", "POSITIVE"] + table.loc["NEGATIVE", "NEGATIVE"] - misread
        tables.append((dept, model, f"{dept.upper()} · {model} · total attribute {n} = {origin} · benar {ok} = "
                                    f"{_pct(ok, n)}%" + (f" · angka salah {misread}" if misread else ""), table))
    return tables


def _save_compare(name: str, tables: list, details: list, rule_counts: dict) -> None:
    """Hasil compare -> Excel <COMPARE_DIR>/<department>/<name>.xlsx (ditimpa bila sudah ada). Sheet "summary": jumlah
    foto, atribut, rule per sub-category, lalu tabel prediksi x actual + akurasi per model. Sheet "ground truth vs
    predicted": isi ground truth (foto x atribut, actual) + kolom predicted_<model>; ada atribut number: juga value
    ground truth dan value_<model> (angka bacaan AI)."""
    for dept in dict.fromkeys(d["department"] for d in details):
        mine = pd.DataFrame([d for d in details if d["department"] == dept])
        has_values = bool((mine["value"] != "").any() or (mine["AI value"] != "").any())
        index = ["subCategoryId", "photo", "folder", "rule", "attribute", "actual"] + (["value"] if has_values else [])
        predicted = (mine.pivot_table(index=index, columns="Model", values="AI", aggfunc="first", sort=False)
                     .add_prefix("predicted_").reset_index())
        if has_values:
            values = (mine.pivot_table(index=index, columns="Model", values="AI value", aggfunc="first", sort=False)
                      .add_prefix("value_").reset_index())
            predicted = predicted.merge(values, on=index, how="left")
        predicted.columns.name = None
        scope = pd.DataFrame([{"subCategoryId": sub, "foto": t["photo"].nunique(), "atribut": t["attribute"].nunique(),
                               "rule": rule_counts[(dept, sub)], "total attribute": len(t)}
                              for sub, t in predicted.groupby("subCategoryId", sort=False)])
        path = COMPARE_DIR / dept / f"{name}.xlsx"
        path.parent.mkdir(parents=True, exist_ok=True)
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            scope.to_excel(writer, index=False, sheet_name="summary")
            row = len(scope) + 3                                   # baris Excel (1-based) judul tabel pertama
            for d, _, title, t in tables:
                if d != dept:
                    continue
                writer.sheets["summary"].cell(row=row, column=1, value=title)
                t.rename_axis(index="Prediction \\ Actual", columns=None).reset_index().to_excel(
                    writer, index=False, sheet_name="summary", startrow=row)
                row += len(t) + 4
            predicted.to_excel(writer, index=False, sheet_name="ground truth vs predicted")
        print(f"saved: {path}")


def compare_ground_truth(rows: list, rules: dict | None = None, save: str | None = None) -> pd.DataFrame:
    """Langkah 2, setelah actual di Excel diperbaiki: positive / negative jawaban AI per atribut (setelah expected dari
    `rules` = setelan rules= yang sama dengan run_exp; None = rule katalog) dibanding kolom actual, per sub-category x
    rule x atribut x model: foto (jumlah foto), n (jumlah penilaian), benar, %, AI positive padahal actual negative, AI
    negative padahal actual positive, angka salah (atribut number: AI dan actual positive, tapi angka AI beda dengan
    kolom value; dihitung salah), foto yang salah. Atribut ber-threshold: positive = angka terbaca dan memenuhi syarat,
    untuk AI (angka AI) dan ground truth (kolom value); actual yang tidak sama dengan threshold dari value = error. Tiap sub-category ditutup baris "semua atribut" per model: total
    semua atributnya (n = foto x atribut, foto salah = foto yang salah di atribut mana pun). actual kosong dilewati.
    Urutan = urutan atribut di jawaban. Sebelum itu tampil tabel prediksi x actual (total attribute) per department x
    model. save: nama bebas (mis. "exp1.3_vs_gt") -> juga disimpan ke hasil-experiment/<department>/<save>.xlsx
    (ditimpa bila sudah ada): sheet summary (jumlah foto / atribut / rule, tabel prediksi x actual + akurasi per
    model) dan ground truth vs predicted (isi ground truth + predicted_<model>).

    Ground truth sekarang satu file per model (ground_truth/<dept>/<sub>__<model>.xlsx), jadi setiap baris dibandingkan
    dengan file ground truth model itu sendiri, bukan satu file yang dipakai bersama semua model."""
    gts, specs, agg, order = {}, {}, {}, {}
    confusion = defaultdict(int)   # (department, model, AI positive, actual positive) -> jumlah foto x atribut
    misreads = defaultdict(int)    # (department, model) -> angka salah
    unchecked = defaultdict(set)   # sub-category -> (foto, atribut) number positive yang value-nya belum diisi
    mismatched = set()             # (foto, atribut, value, actual): actual beda dengan threshold dari value
    scope = defaultdict(lambda: {"photos": set(), "attributes": set()})   # (department, model, sub-category)
    rule_counts, details = {}, []   # details: satu baris per foto x atribut x model (untuk save=)
    for r in rows:
        if r.get("error") or not r.get("facts"):
            continue
        spec_key = (r["departmentId"], r["subCategoryId"])
        gt_key = (*spec_key, r["model"])
        if spec_key not in specs:
            sub_rules = active_rules(*spec_key, rules)
            specs[spec_key], rule_counts[spec_key] = attributes_of(sub_rules), len(sub_rules)
        if gt_key not in gts:
            gts[gt_key] = _load_ground_truth(*gt_key)
        cells = gts[gt_key].get(r["relPath"]) or {}
        for a, f in r["facts"].items():
            cell = cells.get(a)
            if not isinstance(f, dict) or not cell or cell["actual"] is None:
                continue
            got, want = _passed(specs[spec_key], a, f), cell["actual"]
            spec = specs[spec_key].get(a, {})
            if spec.get("threshold") and cell["value"] is not None \
                    and threshold_ok(spec, {"is_true": True, "value": cell["value"]}) != want:
                mismatched.add((r["relPath"], a, cell["value"], want))
            misread = got and want and _misread(specs[spec_key], a, f, cell)
            ok = got == want and not misread
            read = _read_number(spec, f)
            if want and spec.get("type") == "number" and cell["value"] is None:
                unchecked[r["subCategoryId"]].add((r["relPath"], a))
            details.append({"department": r["departmentId"], "subCategoryId": r["subCategoryId"], "photo": r["relPath"],
                            "folder": GROUND_TRUTH.get(r.get("groundTruth"), ""), "rule": cell["rule"], "attribute": a,
                            "actual": "positive" if want else "negative",
                            "value": "" if cell["value"] is None else _number_text(cell["value"]), "Model": r["model"],
                            "AI": "positive" if got else "negative", "AI value": "" if read is None else _number_text(read)})
            confusion[(r["departmentId"], r["model"], got, want)] += 1
            misreads[(r["departmentId"], r["model"])] += misread
            seen = scope[(r["departmentId"], r["model"], r["subCategoryId"])]
            seen["photos"].add(r["relPath"])
            seen["attributes"].add(a)
            order.setdefault((r["subCategoryId"], a), len(order))
            s = agg.setdefault((r["subCategoryId"], cell["rule"], a, r["model"]),
                               {"n": 0, "ok": 0, "fp": 0, "fn": 0, "misread": 0, "wrong": [], "photos": set()})
            s["n"] += 1
            s["photos"].add(r["relPath"])
            s["ok"] += ok
            s["fp"] += got and not want
            s["fn"] += want and not got
            s["misread"] += misread
            if not ok:
                s["wrong"].append(r["image"])
    if mismatched:
        listed = [f"{os.path.basename(p)} / {a}: value {_number_text(v)} -> actual {'negative' if w else 'positive'}"
                  for p, a, v, w in sorted(mismatched, key=lambda m: (_natural_key(m[0]), m[1]))]
        raise ValueError("Atribut ber-threshold: actual harus sama dengan hasil threshold dari kolom value (positive = angka "
                         "memenuhi syarat). Perbaiki actual di ground truth: " + "; ".join(listed))
    if not agg:
        print("Belum ada yang bisa dibandingkan: ground truth sub-category ini belum ada. "
              "Buat dulu: save_ground_truth_draft(rows, model=..., rules=...)")
    keys = sorted(agg, key=lambda k: (k[0], order[(k[0], k[2])], k[3]))
    columns = ["subCategoryId", "rule", "attribute", "Model", "foto", "n", "benar", "% benar",
               "AI positive, actual negative", "AI negative, actual positive", "angka salah", "foto salah"]
    table = pd.DataFrame([[*k, len(agg[k]["photos"]), agg[k]["n"], agg[k]["ok"], _pct(agg[k]["ok"], agg[k]["n"]),
                           agg[k]["fp"], agg[k]["fn"], agg[k]["misread"], agg[k]["wrong"]] for k in keys], columns=columns)
    parts = []
    for sub in dict.fromkeys(k[0] for k in keys):
        t = table[table["subCategoryId"] == sub]
        summary = []
        for model in dict.fromkeys(t["Model"]):
            mine = [agg[k] for k in keys if k[0] == sub and k[3] == model]
            n, ok = sum(s["n"] for s in mine), sum(s["ok"] for s in mine)
            summary.append([sub, "", "semua atribut", model, len(set().union(*(s["photos"] for s in mine))), n, ok,
                            _pct(ok, n), sum(s["fp"] for s in mine), sum(s["fn"] for s in mine),
                            sum(s["misread"] for s in mine), list(dict.fromkeys(p for s in mine for p in s["wrong"]))])
        parts += [t, pd.DataFrame(summary, columns=columns)]
    tables = _confusion_tables(confusion, scope, rule_counts, misreads)
    for *_, title, t in tables:
        print(title)
        display(t)
    for sub, cells_missing in unchecked.items():
        print(f"{sub}: {len(cells_missing)} foto x atribut number positive belum punya value di ground truth, angkanya "
              f"belum dicek (isi kolom value, atau jalankan save_ground_truth_draft untuk mengisinya dari jawaban model)")
    result = pd.concat(parts, ignore_index=True) if parts else table
    if save and details:
        details.sort(key=lambda d: (d["subCategoryId"], d["photo"], order[(d["subCategoryId"], d["attribute"])],
                                    d["Model"]))
        _save_compare(save, tables, details, rule_counts)
    return result

`dump_prompt` (free) prints what the model receives for the same changes as `run_exp`: the prompt, then references and photo in message order (Nova adds "Panggil tool emit_facts dengan hasilnya."). `stage="extract"`: the watermark call · `photo=`: the references for that photo · `show_schema=True`: the output schema.

In [ ]:
def dump_prompt(
    department: str | None = None,
    sub_category: str | None = None,
    stage: str = "facts",
    photo: str | None = None,
    show_schema: bool = False,
    *,
    models: list[str] | None = None,
    rules: dict[str, list[dict]] | None = None,
    prompt: str | None = None,
    render: Callable | None = None,
    schema: Callable | None = None,
    references: dict[str, dict[str, list[str]]] | None = None,
    max_refs: int | None = None,
    exclude_self: bool | None = None,
    exact_references: bool | None = None,
    extract: bool | None = None,
    extract_prompt: str | None = None,
    extract_schema: type[BaseModel] | None = None,
) -> None:
    """Cetak persis apa yang diterima model (teks prompt, urutan gambar, schema) untuk setelan Exp, tanpa memanggil
    model (gratis). Setelan sama dengan run_exp; yang tidak diisi (None) = nilai BASELINE.

    Args:
        department: department sub-category yang dicetak, mis. "dept_a".
        sub_category: id sub-category, mis. "fr-a-02".
        stage: "facts" = prompt fakta; "extract" = prompt ekstraksi watermark (sama untuk semua foto, tanpa sub-category).
        photo: key foto yang dinilai, untuk melihat referensi yang terpilih untuk foto itu (foto itu sendiri dilewati
            bila exclude_self). None = tanpa foto tertentu.
        show_schema: True = cetak juga schema output yang dikirim.
        models: (8.8) id model yang dijalankan, mis. ["nova-pro-bedrock"]. None = MODELS (bagian 1).
        rules: (8.2 / 8.3 / 8.6) {"dept/sub": [rule, ...]} pengganti rule katalog untuk sub-category itu, format
            bagian 8. Sub-category lain tetap rule katalog. None = rule katalog.
        prompt: (8.4) teks template prompt fakta, dengan {sub_category_id} {requirement} {rules}. None = PROMPTS["facts"].
        render: (8.4) fungsi rules -> teks {rules} di prompt. None = render_rules.
        schema: (8.5) fungsi atribut -> model output Pydantic. None = build_output_model.
        references: (8.7) {"dept/sub": {"positive": [key], "negative": [key]}} set referensi pilihan, dicoba paling
            dulu. None = references di katalog.
        max_refs: (8.7) jumlah referensi per role (positive / negative). None = 5.
        exclude_self: (8.7) foto yang dinilai tidak dipakai jadi referensinya sendiri, juga saat exact_references=True
            (set yang dikunci). False hanya untuk uji coba satu foto. None = True.
        exact_references: (8.7) True = set referensi dipakai apa adanya: tanpa tambahan foto folder, tanpa batas
            max_refs. None = False (set dulu, lalu foto folder urut nama).
        extract: (8.9) jalankan ekstraksi watermark (tanggal, lat / long, alamat) sebagai call kedua. None = True.
        extract_prompt: (8.9) teks prompt ekstraksi watermark. None = PROMPTS["extract"].
        extract_schema: (8.9) model output ekstraksi. None = watermark_model().
            (models dan extract tidak berpengaruh di sini.)
    """
    s = settings(**{k: v for k, v in locals().items() if k in BASELINE})   # parameter setelan di atas
    if stage == "extract":
        print("==== prompt · watermark extraction (sama untuk semua foto) ====")
        print(s["extract_prompt"] or PROMPTS["extract"])
        print('(Nova menambahkan di akhir: "\\n\\nPanggil tool emit_extraction dengan hasilnya.")')
        print("==== image ====\n[foto yang dinilai]   (tanpa reference)")
        if show_schema:
            print("==== output schema ====")
            print(json.dumps(bedrock_schema(s["extract_schema"] or watermark_model()), ensure_ascii=False, indent=1))
        return
    _, c = find_category(department, sub_category)
    if not c:
        raise ValueError(f"sub-category {department}/{sub_category} tidak ada di catalog")
    rules = active_rules(department, sub_category, s["rules"])
    if not rules:
        print(f"{department}/{sub_category} belum punya rule: isi lewat rules= (change point 8.2). "
              f"Requirement:\n{c.get('requirement')}")
        return
    check_rules(department, sub_category, rules)
    print(f"==== prompt · {department}/{sub_category} ====")
    print(render_prompt(s["prompt"] or PROMPTS["facts"], c, rules, s["render"]))
    print("==== image, urutan di message ====")
    candidates, limit = _ref_candidates(department, sub_category, photo, s)
    n = 0
    for role, keys in candidates.items():
        for key in keys[:limit]:
            n += 1
            print(f'"Ref {n} ({role}) — {os.path.basename(key)}:"   [image {key}]')
    print(f'"Foto yang dinilai:"   [image {photo or "foto yang dinilai"}]   '
          '(Nova: "Foto yang dinilai. Panggil tool emit_facts dengan hasilnya.")')
    if show_schema:
        print("==== output schema ====")
        print(json.dumps(bedrock_schema((s["schema"] or build_output_model)(attributes_of(rules))), ensure_ascii=False, indent=1))


# Parameter setelan run_exp / reevaluate / dump_prompt harus sama persis dengan BASELINE (tambah setelan = tambah di keempatnya)
for _f in (run_exp, reevaluate, dump_prompt, run_layer2):
    assert set(_f.__kwdefaults__) - {"layer2"} == set(BASELINE), f"{_f.__name__}: parameter setelan beda dengan BASELINE"

In [ ]:
# Examples (baseline): print what the model receives
# dump_prompt("dept_b", "fr-b-01")
# dump_prompt("dept_b", "fr-b-02", show_schema=True)
# dump_prompt(stage="extract")


All sub-categories in the catalog with their photo counts, ids to copy into section 1 or `photos_of`. `phase` empty = undecided; `rules` 0 = no rule yet (all DEPT-B), so `run_exp()` skips them until rules are written (8.2).

In [ ]:
pd.DataFrame([{"DEPARTMENT": d["id"], "MAIN_CATEGORY": mc["id"], "SUB_CATEGORY": c["id"],
               "phase": c.get("phase"), "rules": len(c.get("rules") or []),
               "attributes": len(attributes_of(c.get("rules") or [])),
               "positive": len(c.get("positive") or []), "negative": len(c.get("negative") or []),
               "requirement": c.get("requirement") or ""}
              for d in _load()["departments"] for mc in d["mainCategories"] for c in mc["subCategories"]])

`run_exp()` without `photos=` runs every photo of the sub-categories in the section 1 scope that have rules (table below), counted again at each call. Re-running this cell empties `hasil`.

In [ ]:
PER_SUB = ["departmentId", "mainCategoryId", "subCategoryId"]   # per-sub-category breakdown: run_summary(rows, by=PER_SUB)


def sub_in_scope(department: str, main_category: str, sub: dict) -> bool:
    """Scope from section 1: DEPARTMENT, MAIN_CATEGORY, SUB_CATEGORY, PHASES."""
    return (DEPARTMENT in (None, department) and MAIN_CATEGORY in (None, main_category)
            and SUB_CATEGORY in (None, sub["id"]) and (PHASES is None or sub.get("phase") in PHASES))


def in_scope(task: dict) -> bool:
    """A photo is part of run_exp() without photos= when its sub-category is in scope and has rules in the catalog."""
    _, c = find_category(task["departmentId"], task["subCategoryId"])
    return sub_in_scope(task["departmentId"], task["mainCategoryId"], c) and bool(c.get("rules"))


def _default_photos() -> list[dict]:
    """Photos for run_exp() without photos=, computed at call time (photos added with reload_photos are included)."""
    return [t for t in discover() if in_scope(t)]


_scope = _default_photos()
hasil = {}   # hasil["expN.k"] = rows of one experiment; each run_exp only uses the settings written in its cell (rest = BASELINE)
if not _scope:
    print(f"No photos in scope for DEPARTMENT={DEPARTMENT!r}, MAIN_CATEGORY={MAIN_CATEGORY!r}, SUB_CATEGORY={SUB_CATEGORY!r}, "
          f"PHASES={PHASES!r}. Add photos under notebook/images/<department>/<main>/<sub>/{{positive,negative}}/ and re-run "
          "from the top (or call reload_photos()).")
else:
    print(f"run_exp() without photos=: {len(_scope)} photos x {len(MODELS)} models = {len(_scope) * len(MODELS)} calls")
    display(pd.DataFrame(_scope).groupby(["departmentId", "mainCategoryId", "subCategoryId", "role"]).size().unstack(fill_value=0))


Requirement (the original client sentence, as the prompt gets it), rules and sheet notes of every sub-category in scope: material for writing rules (8.2, 8.3).

In [ ]:
pd.DataFrame([{"SUB_CATEGORY": c["id"], "requirement": c.get("requirement") or "",
               "rules": "; ".join(r["title"] for r in c.get("rules") or []) or "-",
               "notes": c.get("notes") or ""}
              for d in _load()["departments"] for mc in d["mainCategories"] for c in mc["subCategories"]
              if sub_in_scope(d["id"], mc["id"], c)])

### Save Experiment Result to CSV

 Exports run_exp results (list of dicts, e.g. hasil["exp_fr_b_01"]) to CSV files under notebook/hasil-experiment/<department>/, 
 auto-creating the folder if needed. 
 
Usage: `exporter = ResultExporter()` once, 
    then call `exporter.to_csv(hasil["exp_fr_b_01"],"exp_fr_b_01")` from any cell, 
    or `exporter.all_to_csv(hasil)` to export every key in `hasil` at once;
    pass `out_dir="hasil-experiment/dept_a"` to target a different department.
    If `<name>.csv` already exists, it asks `(y/n)` first: only `y` overwrites, otherwise nothing is saved.

`exporter.load("exp_fr_b_01")` reads a CSV back into rows (facts included), so an old run can be evaluated without calling the models again, e.g. `compare_ground_truth(exporter.load("exp1.3"))`.

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime


class ResultExporter:
    """Export hasil run_exp (list of dict) ke CSV di notebook/hasil-experiment/dept_b/, dengan facts di-flatten per kolom."""

    def __init__(self, out_dir: str = "hasil-experiment/dept_b", encoding: str = "utf-8-sig"):
        base = REPO / "notebook"
        self.out_dir = base / out_dir
        self.encoding = encoding
        self.out_dir.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def _flatten_facts(df: pd.DataFrame, facts_col: str = "facts") -> pd.DataFrame:
        """Pecah kolom `facts` (dict per-atribut: evidence/is_true/value) jadi kolom flat:
        <atribut>_evidence, <atribut>_is_true, <atribut>_value."""
        if facts_col not in df.columns:
            return df
        df = df.copy()
        facts_series = df[facts_col].apply(lambda x: x if isinstance(x, dict) else {})
        flat_rows = []
        for facts in facts_series:
            row = {}
            for attr, val in facts.items():
                if isinstance(val, dict):
                    row[f"{attr}_evidence"] = val.get("evidence")
                    row[f"{attr}_is_true"] = val.get("is_true")
                    if "value" in val:
                        row[f"{attr}_value"] = val.get("value")
                else:
                    row[attr] = val
            flat_rows.append(row)
        flat_df = pd.DataFrame(flat_rows, index=df.index)
        return pd.concat([df.drop(columns=[facts_col]), flat_df], axis=1)

    @staticmethod
    def _flatten_other(df: pd.DataFrame) -> pd.DataFrame:
        """Kolom dict/list sisanya (thresholdChecks, references, dll) -> JSON string."""
        df = df.copy()
        for col in df.columns:
            if df[col].apply(lambda x: isinstance(x, (dict, list))).any():
                df[col] = df[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (dict, list)) else x)
        return df

    def _confirm_overwrite(self, path, new_rows: int) -> bool:
        """<name>.csv sudah ada: tanya dulu sebelum ditimpa. Hanya "y" / "yes" yang menimpa; tanpa input (mis. run
        non-interaktif) dianggap tidak."""
        old_rows = len(pd.read_csv(path, dtype=str, keep_default_na=False, encoding=self.encoding))
        changed = datetime.fromtimestamp(path.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
        try:
            answer = input(f"{path.name} sudah ada ({old_rows} rows, diubah {changed}). "
                           f"Timpa dengan {new_rows} rows baru? (y/n): ")
        except (EOFError, NotImplementedError):
            return False
        return answer.strip().lower() in ("y", "yes")

    def to_csv(self, rows: list, name: str, flatten_facts: bool = True) -> str | None:
        """Simpan ke <out_dir>/<name>.csv. Bila file itu sudah ada, tanya dulu (y/n); selain y tidak disimpan (hasil masih
        di memori: simpan lagi dengan nama lain)."""
        path = self.out_dir / f"{name}.csv"
        if path.exists() and not self._confirm_overwrite(path, len(rows)):
            print(f"tidak disimpan: {path.name} tetap seperti sebelumnya. Simpan dengan nama lain, "
                  f'mis. exporter.to_csv(rows, "{name}_b")')
            return None
        df = pd.DataFrame(rows)
        if flatten_facts:
            df = self._flatten_facts(df)
        df = self._flatten_other(df)
        df.to_csv(path, index=False, encoding=self.encoding)
        print(f"saved: {path}  ({len(df)} rows)")
        return str(path)

    NUMBER_COLUMNS = ("promptTokens", "outputTokens", "costUsd", "latencyMs", "refCount", "latitude", "longitude",
                      "layer2CostUsd")

    @staticmethod
    def _cell(text: str, number: bool = False):
        """Sel CSV -> nilai: daftar / dict (yang ditulis _flatten_other sebagai JSON), angka untuk kolom angka; kosong = None."""
        if text == "":
            return None
        if number or text[0] in "[{":
            try:
                return json.loads(text)
            except ValueError:
                pass
        return text

    def load(self, name: str) -> list:
        """Kebalikan to_csv: baris hasil dari <out_dir>/<name>.csv, facts dikembalikan dari kolom <atribut>_is_true /
        _evidence / _value. Untuk menilai run lama tanpa memanggil model lagi, mis.
        compare_ground_truth(exporter.load("exp1.3"))."""
        path = self.out_dir / f"{name}.csv"
        df = pd.read_csv(path, dtype=str, keep_default_na=False, encoding=self.encoding)
        attrs = [c[: -len("_is_true")] for c in df.columns if c.endswith("_is_true")]
        fact_columns = {f"{a}_{part}" for a in attrs for part in ("evidence", "is_true", "value")}
        rows = []
        for rec in df.to_dict("records"):
            facts = {}
            for a in attrs:
                if rec[f"{a}_is_true"] == "":
                    continue                                   # atribut ini tidak ditanya di baris itu
                facts[a] = {"evidence": rec.get(f"{a}_evidence", ""), "is_true": rec[f"{a}_is_true"].lower() == "true"}
                if f"{a}_value" in rec:
                    facts[a]["value"] = self._cell(rec[f"{a}_value"], number=True)
            row = {k: self._cell(v, number=k in self.NUMBER_COLUMNS) for k, v in rec.items()
                   if k not in fact_columns and k != "rawText"}
            rows.append({**row, "rawText": rec.get("rawText", ""), "facts": facts or None})
        print(f"loaded: {path}  ({len(rows)} rows)")
        return rows

    def all_to_csv(self, hasil: dict, flatten_facts: bool = True) -> list[str | None]:
        return [self.to_csv(rows, name, flatten_facts=flatten_facts) for name, rows in hasil.items()]


exporter = ResultExporter(out_dir="hasil-experiment/dept_b")

---
## 8 · Change points

What an Exp can change (nothing here runs; section 9 has worked examples). A change to one sub-category needs only that sub-category run; one every photo goes through (8.4, 8.5, `max_refs`, 8.8, 8.9) needs all of Phase 1 before you rely on it.

| Point | What | Setting (`run_exp` / `reevaluate` / `dump_prompt` / `run_layer2`) | Photos to run | Model call |
|---|---|---|---|---|
| 8.1 | Data: test and reference photos | photos into `notebook/images/…` or `upload_photos` (section 2), then `reload_photos()` (mid-experiment) or re-run from the top | the new ones (returned by `reload_photos`), `photos_of(...)`, or all | yes |
| 8.2 | Rules of a sub-category: add, remove, edit the title / description | `rules={"dept/sub": [rule, ...]}` | that sub-category | yes |
| 8.3 | Attributes of a rule: add, remove, name, description, type, unit | `rules=` (the same list, attributes edited) | that sub-category | yes |
| 8.4 | Prompt: full template text, layout of the rules | `prompt=`, `render=` (copy of `render_rules`) | all in scope | yes |
| 8.5 | Output schema | `schema=` (copy of `build_output_model`), plus a line in `prompt=` for a new field | all in scope | yes |
| 8.6 | The `threshold` of an attribute | `rules=` (`threshold` edited) | that sub-category | no: `reevaluate` |
| 8.7 | Reference photos | `references=`, `max_refs=`, `exclude_self=`, `exact_references=` | one sub-category, or all for `max_refs` | yes |
| 8.8 | Model | `models=` | as needed | yes |
| 8.9 | Watermark extraction (date, lat/long, address) | `extract=`, `extract_prompt=`, `extract_schema=` | all in scope | yes |
| 8.10 | Layer 2: second look at chosen attributes, off until used (section 9, Exp 4) | `run_layer2(rows, layer2={"dept/sub": {"recheck": {name: answer}, "prompt": ...}})`, on top of `run_exp` rows | the rows given; only those with the chosen answer | yes |

To change a function, copy it into the Exp cell under a new name, edit it, and pass it (`render=`, `schema=`, `extract_schema=`).

**Rule format** (catalog and `rules=`):

| Field | Meaning |
|---|---|
| `title`, `description` | rule name (unique, numbered in the prompt) and what it checks; both go into the prompt |
| `attributes[].name` | answer field name, `snake_case`; the same name in two rules is asked once |
| `attributes[].description` | what the model looks for and what `is_true` = true means; matters most for accuracy |
| `attributes[].type` | `bool` (true / false only), `text` (+ the text read), `number` (+ the number read, `unit`, e.g. `"dBm"`), `json` (+ one item per object seen) |
| `attributes[].fields` | `json` only: what each item holds, a list of `{name, type: bool / text / number, description, unit?}` |
| `attributes[].threshold` | optional, code only: `{">=": -24}`, `{">=": 200, "<=": 240}`; `number` only (a count is its own `number`). Not met → `thresholdFailed` |

Examples by type: sample catalog FR-B-01 (`bool`, `text`), FR-B-02 (`number` + threshold), FR-B-03 (`json`); a threshold change is Exp 3 in section 9.

In an Exp cell, write the list out (Exp 2 in section 9), or edit a copy: `R = active_rules(dept, sub)`, then `attributes_of(R)[name]` (an attribute), `rule_of(R, title)` (a rule), `R.append({...})` (a new rule). `pprint.pp(active_rules(dept, sub), sort_dicts=False, width=120)` prints rules ready to paste. Kept for good: into `catalog_fact_phase1.json` (commit), then re-run. Never change a sub-category `id`: it is the bucket folder.

In [ ]:
# Rule-format examples from the sample catalog:
# FR-B-03: a `json` attribute (one item per label, with the fields code + format_sesuai)
# FR-B-02: `number` attributes with thresholds (nilai_rsl >= -24 dBm, panjang_gelombang = 1490 nm)
pprint.pp(active_rules("dept_b", "fr-b-03"), sort_dicts=False, width=120)
pprint.pp(active_rules("dept_b", "fr-b-02"), sort_dicts=False, width=120)


**Attribute `type`.** Every attribute answers true / false (`is_true`: its `description` holds for the photo or not); `type` says what the AI reads besides it:

| `type` | The AI answers | Use when | Example |
|---|---|---|---|
| `bool` | `is_true` | yes / no is enough | `retak` → `is_true: false` |
| `text` | `is_true` + `value` (text) | something written must be read | `id_pole_terlihat` → `"JKT-0213"` |
| `number` | `is_true` + `value` (number), `unit`, optional `threshold` | a number must be read | `nilai_rsl` → `-21.4` (dBm), `{">=": -24}` |
| `json` | `is_true` + `value` (one item per object seen), `fields` = what each item holds | several objects, each with its own details | `label_perangkat` → `[{"kode": "ABC-001", "format_sesuai": true}, …]` |

A `json` attribute lists its `fields` like small attributes: `[{"name": "kode", "type": "text", "description": "kode label seperti yang tercetak"}, {"name": "format_sesuai", "type": "bool", "description": "format kode sesuai standar penomoran"}]` (field types `bool` / `text` / `number` + `unit`). A changed `type` changes the schema, so it needs `run_exp` (8.3). The next cell shows a change to `text` and to `json`, free.

In [ ]:
# # Example: changing an attribute `type` (change point 8.3). Free: only the prompt lines and the schema are printed.
# CONTOH_TYPE = copy.deepcopy(active_rules("dept_b", "fr-b-01"))
# # bool -> text: besides true / false, the model writes what it reads / sees in `value`
# attributes_of(CONTOH_TYPE)["penanda_terlihat"].update(
#     type="text", description="penanda berwarna terlihat; value = di bagian mana penanda itu terlihat")
# # bool -> json: `value` = a list, one item per object seen; `fields` = what each item holds (bool / text / number + unit)
# attributes_of(CONTOH_TYPE)["closure_underground"].update(
#     type="json", description="closure terlihat; value = satu item per closure",
#     fields=[{"name": "lokasi", "type": "text", "description": "di mana closure terpasang"},
#             {"name": "tertutup", "type": "bool", "description": "penutup closure terpasang"}])
# check_rules("dept_b", "fr-b-01", CONTOH_TYPE)              # format is checked first, as in run_exp
# print(render_rules(CONTOH_TYPE))                          # the {rules} part of the prompt
# _schema = bedrock_schema(build_output_model(attributes_of(CONTOH_TYPE)))
# print(json.dumps({k: _schema["properties"][k] for k in ("penanda_terlihat", "closure_underground")}, ensure_ascii=False, indent=1))


**How rules reach the prompt.** `{rules}` in the template (`render_rules`, section 3): numbered rules, one `- name: description` line per attribute. FR-B-01 from the sample catalog (`dump_prompt("dept_b", "fr-b-01")` prints it, free):

```text
Rule 1 · Jenis closure
  Closure terpasang di pole (aerial) atau di dalam handhole (underground)
  - jenis_closure: jenis closure; value = aerial atau underground (tulis teks yang terbaca di value)
Rule 2 · Penanda posisi
  Closure aerial ditandai dengan penanda berwarna atau panah pada pole / kabel
  - closure_underground: closure terpasang underground (di dalam handhole), bukan menggantung pada pole / kabel
  - penanda_terlihat: penanda berwarna terlihat pada pole / kabel yang membawa closure. Isi false jika closure underground
```

The prompts and rule text are in Bahasa Indonesia on purpose: the photos, labels and watermarks are Indonesian, and the experiments were run in that language.


---
## 9 · Experiments

Start here; sections 1–8 are the machinery. One Exp = one topic (a sub-category, an attribute, the prompt, …), with any number of change points. The minimum:

```text
### Exp 3 · FR-B-02 (optical reading): thresholds (8.6)                 <- the topic and its change points
#### Exp 3.1 · Run with the catalog thresholds                          <- the part that changes
    hasil["exp3.1"] = await run_exp(photos=photos_of("dept_b", sub="fr-b-02"))
    **Result:** ...
#### Exp 3.2 · Loosen a threshold, no new model call                    <- next attempt, same Exp
    hasil["exp3.2"] = reevaluate(hasil["exp3.1"], rules={"dept_b/fr-b-02": RULES_X})
    **Result:** ...  **Decision:** ...
```

- `Exp N` = topic + change points, `Exp N.k` = attempt (from `.1`). The same part again: re-run the same `.k` (replaced) or open the next `.k` (both kept).
- A run uses only the settings written in its call; anything not given = `BASELINE` (section 7). An attempt that builds on an earlier one repeats its settings plus the change. Results: `hasil["expN.k"]`. Settings can also be a dict, `expN_k = dict(...)`, passed as `run_exp(..., **expN_k)`.
- `rules` / `references` are dicts per sub-category; several at once: `rules={"dept/sub": RULES_X, "dept/sub2": RULES_Y}`.
- Comparing is yours: read the earlier cells, or `compare_attributes`. Decide once per `Exp N`, after its last attempt; a losing attempt needs nothing undone.
- `dump_prompt(..., rules=...)` shows what will be sent, free, before a paid run.

Combinations (all combine freely; `RULES_NEW`, `PROMPT_NEW` = full values written in the cell):

| # | Combination | Call |
|---|---|---|
| 1 | One sub-category's rules, **one photo**, exact reference set | `run_exp(photos=[key], rules={sub: RULES_NEW}, references={sub: SET}, exact_references=True, exclude_self=False)` |
| 2 | One sub-category's rules / attributes, **its photos** | `run_exp(photos=photos_of(dept, sub=s), rules={sub: RULES_NEW})` |
| 3 | Threshold changed or removed: **free** | `reevaluate(hasil["expN.k"], rules={...})` |
| 4 | Prompt, **everything in scope** | `run_exp(prompt=PROMPT_NEW)` |
| 5 | Schema (+ a prompt line for the new field) | `run_exp(schema=build_output_model_new, prompt=PROMPT_NEW)` |
| 6 | Several sub-categories + prompt | `run_exp(rules={sub1: RULES_1, sub2: RULES_2}, prompt=PROMPT_NEW)` |
| 7 | References, model, extraction | `run_exp(max_refs=3)` · `run_exp(models=["nova-pro-bedrock"])` · `run_exp(extract_prompt=TEXT)` · `run_exp(extract=False)` |
| 8 | Everything at once | `run_exp(rules=..., prompt=..., schema=..., references=..., max_refs=..., models=...)` |
| 9 | New photos | into the folder (8.1), `reload_photos()` (prints their keys), then `photos=[key]`, `photos_of(...)` or any of the above |
| 10 | Layer 2 on chosen attributes (off until used) | `run_layer2(hasil["expN.k"], layer2={sub: {"recheck": {name: answer}}}, rules=...)` |

- **Free or paid**: only a threshold change is free (`reevaluate`); anything the model is asked or shown needs `run_exp`.
- **Own reference**: only in a one-photo try-out (no. 1); in a batch `exclude_self=True` (default) keeps the answer hidden.
- **Local or global**: one sub-category's rules or references → run that sub-category; prompt, schema, `max_refs`, model → run all of Phase 1 before relying on it.
- **No verdict yet**: judge an Exp on the attributes it meant to change (`selisih` in `attribute_breakdown`).

The cells below are worked examples on the synthetic sample catalog. They call the models (paid) only when `RUN_PAID = True`.


In [ ]:
RUN_PAID = False   # True = the cells below call the models (cost). Needs photos in notebook/images/ and Bedrock credentials.


### Exp 1 · FR-B-01 baseline

Catalog rules, no references, both models. Reading: `run_summary` for errors and cost, `attribute_breakdown` for the gap between positive and negative photos per attribute.

In [ ]:
if RUN_PAID:
    hasil["exp1.1"] = await run_exp(
        photos=photos_of("dept_b", sub="fr-b-01"),
        references={"dept_b/fr-b-01": {"positive": [], "negative": []}},   # no reference photos: zero-shot baseline
    )
    display(run_summary(hasil["exp1.1"]))
    display(attribute_breakdown(hasil["exp1.1"]))


### Exp 2 · FR-B-01 sharper attribute descriptions (8.3)

A typical iteration: the model confused "marker on the pole" with "marker at the top of the pole", and picked the wrong pole when several are in the photo. The fix is in the attribute description, not in the code. `dump_prompt` shows the result before any paid run.

In [ ]:
RULES_FR_B_01_V2 = [
    {
        "title": "Jenis closure",
        "description": "Closure bisa dikenali sebagai salah satu dari: underground (di dalam handhole), aerial (di pole), "
                       "atau digantung di kabel (memakai pengait / hanger)",
        "attributes": [
            {"name": "jenis_closure", "type": "text",
             "description": "jenis closure bisa dikenali; value = aerial, underground, atau gantung_di_kabel"},
        ],
    },
    {
        "title": "Penanda posisi",
        "description": "Hanya berlaku untuk closure yang menggantung (aerial / gantung_di_kabel): pole ditandai dengan "
                       "penanda berwarna, atau ada panah / caption pada pole / kabel yang sama. Tidak berlaku untuk underground.",
        "attributes": [
            {"name": "closure_underground", "type": "bool",
             "description": "closure terpasang underground (di dalam handhole), bukan menggantung pada pole / kabel"},
            {"name": "penanda_di_ujung_pole", "type": "bool",
             "description": "penanda berwarna terlihat PADA UJUNG pole (bukan di sepanjang batang); bila ada beberapa pole, "
                            "yang dicek adalah pole yang membawa closure. Isi false jika closure underground"},
            {"name": "penanda_panah_atau_caption", "type": "bool",
             "description": "panah atau caption (garis / lingkaran berwarna) terlihat pada pole / kabel yang sama; "
                            "isi false jika closure underground"},
        ],
    },
]

dump_prompt("dept_b", "fr-b-01", rules={"dept_b/fr-b-01": RULES_FR_B_01_V2})   # free: prints exactly what the model receives


In [ ]:
if RUN_PAID:
    hasil["exp2.1"] = await run_exp(photos=photos_of("dept_b", sub="fr-b-01"), rules={"dept_b/fr-b-01": RULES_FR_B_01_V2})
    display(compare_attributes({"exp1.1": hasil["exp1.1"], "exp2.1": hasil["exp2.1"]}))   # gap per attribute, side by side


### Exp 3 · FR-B-02 thresholds (8.6)

`number` attributes are read by the model and checked by code. Changing a threshold needs no new model call: `reevaluate` re-checks the stored facts.

In [ ]:
if RUN_PAID:
    hasil["exp3.1"] = await run_exp(photos=photos_of("dept_b", sub="fr-b-02"))

    RULES_FR_B_02_LOOSER = copy.deepcopy(active_rules("dept_b", "fr-b-02"))
    attributes_of(RULES_FR_B_02_LOOSER)["nilai_rsl"]["threshold"] = {">=": -25}   # was >= -24
    hasil["exp3.2"] = reevaluate(hasil["exp3.1"], rules={"dept_b/fr-b-02": RULES_FR_B_02_LOOSER})   # prints which photos changed


### Exp 4 · Layer 2 on a weak attribute (8.10)

A second look at attributes that are often misread. Only photos whose layer-1 answer matches `recheck` are sent again, and only for those attributes.

In [ ]:
# if RUN_PAID:
#     hasil["exp4.1"] = await run_layer2(
#         hasil["exp2.1"],
#         layer2={"dept_b/fr-b-01": {"recheck": {"penanda_di_ujung_pole": False}}},
#         rules={"dept_b/fr-b-01": RULES_FR_B_01_V2},
#     )


### Ground truth and comparison

After an experiment: `save_ground_truth_draft(rows, model=...)` writes one Excel file per sub-category and model with the model answers as a draft; correct `actual` by hand; then `compare_ground_truth(rows)` reports the accuracy per rule × attribute × model and the confusion table. Results are saved with `exporter.to_csv(rows, "name")`.

In [ ]:
# if RUN_PAID:
#     save_ground_truth_draft(hasil["exp2.1"], model="kimi-k2-5-bedrock", rules={"dept_b/fr-b-01": RULES_FR_B_01_V2})
#     # ... correct the `actual` column in notebook/ground_truth/dept_b/fr-b-01__kimi-k2-5-bedrock.xlsx ...
#     compare_ground_truth(hasil["exp2.1"], rules={"dept_b/fr-b-01": RULES_FR_B_01_V2}, save="exp2.1_vs_gt")
#     exporter.to_csv(hasil["exp2.1"], "exp2.1")


---
## Bounding box · evidence boxes per attribute

Boxes come after the facts. `add_boxes(rows, rules=...)` adds boxes to `run_exp` rows, or to rows loaded with `exporter.load(...)`, without running the facts again; `run_exp_boxes(photos, **settings)` = `run_exp` + `add_boxes`. `show_boxes(rows)` draws the photo with boxes (green = pass, red = fail) and a table per photo × model (`AI` = `true / false (pass / fail)`). To compare with ground truth, pass one photo's `gt` (from `_load_ground_truth`) to `draw_boxes` / `box_table`.


In [ ]:
# Bounding boxes (evidence box per attribute): a second model call after the facts call, grounded on the facts.
# Coordinates are 0-1000 relative to the photo width / height.
import asyncio, base64, io, os, time
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as IPImage

BEDROCK_IDS = {"kimi-k2-5-bedrock": os.environ.get("KIMI_MODEL_ID", "moonshotai.kimi-k2.5"),
               "nova-pro-bedrock": os.environ.get("NOVA_MODEL_ID", "amazon.nova-pro-v1:0")}
JSON_ONLY = ('\n\nKembalikan JSON saja, tanpa penjelasan: {"detections": [{"attribute": "...", "label": "...", '
             '"bbox": [x1, y1, x2, y2], "confidence": 0-100}]}')


class Detection(BaseModel):
    attribute: str = Field(description="nama atribut, mis. box_ont_terlihat")
    label: str = Field(description="nama objek singkat")
    bbox: list[int] = Field(description="[x1, y1, x2, y2], skala 0-1000 terhadap lebar dan tinggi foto")
    confidence: int = Field(description="keyakinan 0-100")


class Grounding(BaseModel):
    detections: list[Detection]


def render_box_rules(rules: list) -> str:
    """{rules} prompt kotak: susunan render_rules (prompt fakta), atribut cukup nama: description, explanation contoh
    objek dikutip apa adanya (call kotak tidak mengirim gambar contoh). Contoh objek tanpa explanation tidak menambah teks."""
    lines, asked = [], set()
    for i, r in enumerate(rules, 1):
        lines.append(f"Rule {i} · {r['title']}")
        if r.get("description"):
            lines.append(f"  {r['description']}")
        for a in r.get("attributes") or []:
            if a["name"] in asked:
                lines.append(f"  - {a['name']}: (sama dengan di atas)")
                continue
            asked.add(a["name"])
            lines.append(f"  - {a['name']}: {a['description']}")
            notes = [p["explanation"] for p in a.get("objectPhotos") or [] if p.get("explanation")]
            if notes:
                lines.append("    Penjelasan contoh objek dari rule: " + " ".join(f'"{n}"' for n in notes))
    return "\n".join(lines)


def ground_photo(model_id: str, rel_path: str, prompt: str) -> dict:
    """Satu foto x satu model -> kotak per atribut (0-1000) + token, biaya, latency. Nova: tool emit_boxes. Kimi: JSON
    teks (dengan structured output Kimi mengisi kotak [0, 0, 0, 0] di 3 dari 4 foto uji)."""
    import boto3
    from botocore.config import Config
    t0, nova = time.time(), model_id.startswith("nova")
    b64, mime = encode_photo(rel_path)
    content = [{"image": {"format": mime.split("/")[-1], "source": {"bytes": base64.b64decode(b64)}}},
               {"text": prompt if nova else prompt + JSON_ONLY}]
    client = boto3.client("bedrock-runtime", region_name=os.environ.get("AWS_REGION", "ap-southeast-3"),
                          config=Config(retries={"max_attempts": 10, "mode": "adaptive"}, read_timeout=300))
    kw = dict(modelId=BEDROCK_IDS[model_id], messages=[{"role": "user", "content": content}],
              inferenceConfig={"maxTokens": 2048, "temperature": VERDICT_TEMPERATURE})
    if nova:
        kw["toolConfig"] = {"tools": [{"toolSpec": {"name": "emit_boxes", "description": "Laporkan bounding box per atribut.",
                                                    "inputSchema": {"json": bedrock_schema(Grounding)}}}],
                            "toolChoice": {"tool": {"name": "emit_boxes"}}}
    try:
        resp = client.converse(**kw)
    except Exception as e:
        return {"relPath": rel_path, "model": model_id, "detections": None, "raw": "", "error": f"{type(e).__name__}: {e}",
                "latencyMs": int((time.time() - t0) * 1000)}
    if nova:
        raw = next((b["toolUse"]["input"] for b in resp["output"]["message"]["content"] if "toolUse" in b), {})
        g = parse_dict(Grounding, raw)
    else:
        raw = _text_of(resp)
        g = parse_json(Grounding, raw)
    u = resp.get("usage", {})
    tu = {"promptTokens": u.get("inputTokens", 0), "outputTokens": u.get("outputTokens", 0)}
    return {"relPath": rel_path, "model": model_id, "detections": [d.model_dump() for d in g.detections] if g else None,
            "raw": "" if g else str(raw), "error": "", **tu, "costUsd": _cost_usd(model_id, tu),
            "latencyMs": int((time.time() - t0) * 1000)}


def draw_boxes(row: dict, facts: dict | None = None, rules: list | None = None, gt: dict | None = None) -> Image.Image:
    """Kotak di gambar yang sama dengan yang dikirim ke model (encode_photo), 0-1000 -> piksel. Dengan facts + rules:
    warna = hasil atribut setelah expected (hijau pass, merah fail); dengan gt: "| GT: ..." kalau ground truth beda.
    Atribut tanpa kotak tidak ditulis di gambar (lihat box_table). Hanya untuk dilihat (foto customer); foto yang lebarnya
    di bawah 640 px diperbesar untuk tampilan saja, supaya label terbaca (koordinat 0-1000 tetap pas)."""
    b64, _ = encode_photo(row["relPath"])
    im = Image.open(io.BytesIO(base64.b64decode(b64))).convert("RGB")
    if im.width < 640:
        im = im.resize((640, round(im.height * 640 / im.width)))
    draw, (w, h) = ImageDraw.Draw(im), im.size
    try:
        font = ImageFont.load_default(size=max(14, w // 55))
    except TypeError:   # Pillow < 10.1
        font = ImageFont.load_default()
    specs = attributes_of(rules) if rules else {}

    def result(name):   # "pass" / "fail" / None, ditambah GT kalau beda (juga angka yang beda dengan kolom value)
        if facts is None or name not in specs:
            return None, ""
        ai = "pass" if _passed(specs, name, facts.get(name)) else "fail"
        cell = (gt or {}).get(name) or {}
        actual = cell.get("actual")
        if actual is None:
            return ai, ""
        if ("pass" if actual else "fail") != ai:
            return ai, f" | GT: {'pass' if actual else 'fail'}"
        if actual and _misread(specs, name, facts.get(name), cell):
            return ai, f" | GT: {_number_text(cell['value'])}"
        return ai, ""

    labels = []
    for det in row.get("detections") or []:
        if len(det["bbox"]) != 4:
            continue
        x1, y1, x2, y2 = (v / 1000 * s for v, s in zip(det["bbox"], (w, h, w, h)))
        ai, ref = result(det["attribute"])
        color = {"pass": "#2ca02c", "fail": "#d62728"}.get(ai, "#1f77b4")   # hijau pass, merah fail, biru = tanpa fakta
        draw.rectangle([x1, y1, x2, y2], outline=color, width=max(5, w // 200) * (2 if ref else 1))   # beda GT: lebih tebal
        text = f'{det["attribute"]}: {ai} · {det["confidence"]}{ref}' if ai else f'{det["attribute"]} ({det["confidence"]})'
        left, top, right, bottom = draw.textbbox((0, 0), text, font=font)
        tw, th = right - left + 6, bottom - top + 6
        tx = max(0, min(x1, w - tw))                       # label tidak keluar dari sisi kanan
        if y1 - th >= 0:
            ty = y1 - th                                   # di atas kotak
        elif y2 + 2 + th <= h:
            ty = y2 + 2                                    # di bawah kotak
        else:
            ty = max(0, y1)                                # kotak setinggi foto: di dalam kotak, bagian atas
        while any(tx < c and tx + tw > a and ty < d and ty + th > b for a, b, c, d in labels):
            ty += th   # label kotak lain di posisi yang sama: geser ke bawah
        labels.append((tx, ty, tx + tw, ty + th))
        draw.rectangle([tx, ty, tx + tw, ty + th], fill=color)
        draw.text((tx + 3, ty + 1), text, fill="white", font=font)
    return im


def _seen(specs: dict, name: str, fact: dict | None) -> bool:
    """Atribut terlihat, atau angkanya terbaca, menurut fakta (true / false di tabel = "ya" / "tidak" di prompt kotak):
    aturan _passed dengan threshold dilepas dan expected true, jadi selalu sejalan dengan _passed (warna kotak, ground
    truth) tanpa rumus kedua."""
    return _passed({name: {**specs.get(name, {}), "threshold": None, "expected": True}}, name, fact)


def box_table(row: dict, facts: dict | None = None, rules: list | None = None, gt: dict | None = None) -> pd.DataFrame:
    """Per atribut satu foto: AI dan GT masing-masing satu kolom "true (pass)" / "false (fail)". true / false = atribut
    terlihat atau angkanya terbaca (_seen, sama dengan "ya" / "tidak" di prompt kotak); pass / fail = setelah threshold
    dan expected (_passed), juga warna kotak. Atribut cacat (expected false): "true (fail)" = cacatnya terlihat. Angka
    terbaca tapi gagal threshold: "true (fail · threshold)". Ground truth menyimpan pass / fail, jadi kolom GT atribut
    ber-threshold berisi pass / fail. Atribut number: angka bacaan AI dan kolom value ground truth ditulis di belakang
    (" · -19.34"); angka yang beda dihitung salah. Lalu benar/salah, jumlah kotak, confidence kotak, dan evidence AI
    (alasan jawabannya). AI true, GT false = AI melihat sesuatu yang tidak ada; sebaliknya = AI melewatkannya. Foto yang
    belum ada di ground truth (gt kosong): tanpa kolom GT dan benar."""
    specs = attributes_of(rules) if rules else {}
    boxes = {}
    for d in row.get("detections") or []:
        boxes.setdefault(d["attribute"], []).append(d)
    out = []
    for name, spec in specs.items():
        expected, threshold = spec.get("expected", True), spec.get("threshold")
        fact = None if facts is None else facts.get(name)
        ai = None if facts is None else _passed(specs, name, fact)
        cell = (gt or {}).get(name) or {}
        actual = cell.get("actual")
        verdict = lambda p: "pass" if p else "fail"
        number = lambda v: "" if v is None else f" · {_number_text(v)}"   # angka atribut number
        ai_text = None
        if ai is not None:
            seen = _seen(specs, name, fact)
            ai_text = (f"{str(seen).lower()} ({verdict(ai)}{' · threshold' if threshold and seen and not ai else ''})"
                       + number(_read_number(spec, fact)))
        gt_text = None
        if actual is not None:                             # threshold: terbaca atau tidak, tidak diketahui dari GT
            gt_text = (verdict(actual) if threshold else f"{str(actual == expected).lower()} ({verdict(actual)})") \
                + number(cell.get("value"))
        wrong = ai != actual or (ai and actual and _misread(specs, name, fact, cell))
        line = {"attribute": name, "expected": expected, "AI": ai_text, "GT": gt_text,
                "benar": "" if ai is None or actual is None else ("✗" if wrong else "✓"),
                "kotak": len(boxes.get(name, [])), "confidence": ", ".join(str(d["confidence"]) for d in boxes.get(name, [])),
                "evidence": ((facts or {}).get(name) or {}).get("evidence", "")}
        if not gt:                                         # belum ada ground truth: kolomnya pasti kosong
            line = {k: v for k, v in line.items() if k not in ("GT", "benar")}
        out.append(line)
    return pd.DataFrame(out)


def render_facts(facts: dict, rules: list) -> str:
    """Hasil call fakta per atribut: ya / tidak (_seen: terlihat, atau angkanya terbaca; threshold tidak ikut) + evidence."""
    specs, lines = attributes_of(rules), []
    for name in specs:
        f = facts.get(name) or {}
        lines.append(f"- {name}: {'ya' if _seen(specs, name, f) else 'tidak'}. Evidence: {f.get('evidence', '')}")
    return "\n".join(lines)


def display_jpeg(im: Image.Image) -> None:
    """Tampilkan gambar sebagai JPEG (±7x lebih kecil dari PNG bawaan Jupyter), supaya notebook tidak terlalu besar untuk
    disimpan (JupyterHub: "Failed to fetch" saat notebook ±30 MB)."""
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=85)
    display(IPImage(data=buf.getvalue(), format="jpeg"))


def show_boxes(rows: list, rules: dict | None = None) -> None:
    """Hasil AI saja, dilihat sebelum ground truth: gambar berkotak (warna = pass / fail atribut menurut AI) + tabel
    attribute, expected, AI, kotak, confidence, evidence per baris (foto x model). Output berisi foto. Dibanding ground
    truth: oper gt= satu foto ke draw_boxes / box_table, dengan cara ground truth notebook masing-masing (DEPT-C:
    show_boxes_vs_gt)."""
    for r in rows:
        view = dict(facts=r.get("facts"), rules=active_rules(r["departmentId"], r["subCategoryId"], rules))
        print(r["model"], "|", r["image"], "| error:", r.get("error") or r.get("boxError") or "-")
        display_jpeg(draw_boxes(r, **view))
        display(box_table(r, **view))


BOX_PROMPT_3 = """Anda membantu auditor QC instalasi jaringan telekomunikasi. Sub-category: {sub_category_id}.
Requirement: {requirement}

Rule untuk sub-category ini, masing-masing dengan atribut yang diamati:
{rules}

Hasil pemeriksaan foto ini (dari call fakta):
{facts}

Tugas: beri bounding box pada bagian FOTO yang menjadi bukti tiap atribut, menurut jenis atributnya dan hasil pemeriksaan di atas:
- Keberadaan objek: "ya" -> kotaki objeknya, satu kotak per objek utuh (bukan per bagian); "tidak" -> tanpa kotak.
- Kondisi orang atau objek: "ya" atau "tidak" -> satu kotak di bagian yang menunjukkan kondisi itu.
- Cacat: "ya" -> kotaki area cacatnya; "tidak" -> tanpa kotak.
- Nilai (nomor, teks, angka): "ya" -> kotaki tempat nilai itu terbaca; "tidak" -> tanpa kotak.
- Objek yang terlihat tetap dikotaki, apa pun kondisi fotonya.
- Tidak yakin letaknya: tanpa kotak. Jangan menebak.
Format: [x1, y1, x2, y2], skala 0-1000 terhadap lebar dan tinggi foto. Kotak serapat mungkin."""


async def add_boxes(rows: list, rules: dict | None = None, box_prompt: str | None = None) -> list:
    """Call kotak setelah fakta untuk baris run_exp atau exporter.load: satu call per baris yang faktanya ada, hanya foto
    yang dinilai, dengan hasil fakta baris itu di prompt (render_facts). rules: setelan rules= yang sama dengan run_exp
    (None = rule katalog). box_prompt: template dengan {sub_category_id} {requirement} {rules} {facts}; None =
    BOX_PROMPT_3. Kolom tambahan: detections, boxError, boxCostUsd, boxLatencyMs; costUsd = fakta + kotak."""
    sems = {m: asyncio.Semaphore(CONCURRENCY.get(m, 4)) for m in {r["model"] for r in rows}}

    async def one(r: dict) -> dict:
        if not r.get("facts"):
            return {**r, "detections": None, "boxError": "tanpa fakta, kotak tidak dicari"}
        dept, sub = r["departmentId"], r["subCategoryId"]
        _, c = find_category(dept, sub)
        sub_rules = active_rules(dept, sub, rules)
        prompt = (box_prompt or BOX_PROMPT_3).format(
            sub_category_id=sub, requirement=(c or {}).get("requirement", ""), rules=render_box_rules(sub_rules),
            facts=render_facts(r["facts"], sub_rules))
        async with sems[r["model"]]:
            g = await asyncio.to_thread(ground_photo, r["model"], r["relPath"], prompt)
        error = g["error"] or ("" if g["detections"] is not None else f"output kotak tidak sesuai schema: {g['raw'][:300]}")
        return {**r, "detections": g["detections"], "boxError": error, "boxCostUsd": g.get("costUsd"),
                "boxLatencyMs": g["latencyMs"], "costUsd": (r.get("costUsd") or 0) + (g.get("costUsd") or 0)}

    out = list(await asyncio.gather(*(one(r) for r in rows)))
    asked = [r for r in out if r.get("facts")]
    print(f"{len(asked)} call kotak, {sum(1 for r in asked if r['boxError'])} error, "
          f"{len(out) - len(asked)} baris tanpa fakta dilewati")
    return out


async def run_exp_boxes(photos: list | None = None, **options) -> list:
    """run_exp (fakta, setelan apa pun: rules=, prompt=, references=, ...) lalu add_boxes dengan rules= yang sama.
    Berurutan per batch: semua call fakta dulu, baru semua call kotak."""
    return await add_boxes(await run_exp(photos, **options), rules=options.get("rules"))


In [ ]:
# Boxes for facts that already exist: no new facts run, only the box call (about $0.002-0.003 per photo x model).
# rules= must match the run that produced the CSV (None = catalog rules). Output contains photos: clear outputs before committing.
# hasil["exp2.1_boxes"] = await add_boxes(exporter.load("exp2.1"), rules={"dept_b/fr-b-01": RULES_FR_B_01_V2})
# exporter.to_csv(hasil["exp2.1_boxes"], "exp2.1_boxes")
# show_boxes(hasil["exp2.1_boxes"][:2], rules={"dept_b/fr-b-01": RULES_FR_B_01_V2})   # 2 rows first: photo with boxes + table
